# Where should the ANN/SNN boundary sit in a hybrid network — and how much energy does that placement actually save?

*A hardware-aware, reproducible study on spoken-digit audio.*
*A high-school science-fair project — written to be read critically.*

Hi. I'm a high-school student, and this notebook is my attempt to answer one question honestly:

> If a network has to be **part ANN** (the ordinary dense, multiply-based kind) and **part SNN** (the spiking, event-based kind), **where should the seam go** — and how much energy does moving that seam actually save?

I train **four families of network** on **two different representations of the same spoken digits**, count every operation from real execution traces, and estimate energy under **two independent energy models**. Then I compare everything.

I want to be upfront about what this is and isn't, because being clear about the limits is the most important part of doing this well.

**What this is.** A careful, fully-audited *replication* on a small task I can actually run and defend. Every number about *operations* is an exact measurement from a forward pass. Every number about *energy* is a model — clearly labelled as a model. The accounting follows the strongest hardware-aware paper I could find: Dampfhoffer et al. (2023), provided as a PDF alongside this notebook.

**What this is not.** It's not a hardware measurement — I don't have a neuromorphic chip, so nothing here was measured on one. It's not a new architecture, and it's not a new record. Hybrid ANN-SNN is a crowded, active research area (I list the neighbours in §3). My contribution is *clarity and honesty of the accounting*, not novelty.

> **How I use sources.** In an earlier version of this project, most of my mistakes came from repeating numbers I'd seen in search snippets without reading the actual paper. So now every external claim carries one of three tags:
>
> - **[READ]** — I opened and read the full source while preparing this notebook.
> - **[ABSTRACT]** — I read only the abstract or publisher page (full text paywalled).
> - **[UNREAD]** — I never accessed it; treat any number tagged this way as unverified.
>
> Wherever I can, I link the source so you can check me.


## Jargon, defined once (please read this before the results)

This field is full of abbreviations that rarely get spelled out, and none of my results make sense without them. Here they are in plain language. I assume you know calculus and basic neural networks, but nothing about spiking hardware.

**ANN — Artificial Neural Network.** The ordinary kind. Each layer multiplies inputs by weights, sums, applies a nonlinearity. Every input touches every weight every time. Its atomic operation is a **MAC**.

**SNN — Spiking Neural Network.** Neurons talk in binary **spikes** (a 1, or nothing). A neuron adds up incoming weights only when a spike arrives, stores the running total in a **membrane potential**, and fires its own spike when that potential crosses a threshold. Its atomic operation is an **AC**. A silent neuron costs almost nothing on the right hardware.

**MAC — Multiply-Accumulate.** `out += weight × input`. One multiply, one add. The workhorse of an ANN.

**AC — Accumulate.** `out += weight`. Just an add — no multiply, because the input is a spike (a 1), so there's nothing to multiply by. Cheaper than a MAC.

**LIF — Leaky Integrate-and-Fire.** The spiking neuron I use throughout. Its membrane potential *decays* toward zero a little each timestep (it "leaks"). This is the standard neuron and — importantly for me — it's the one actually built into commercial neuromorphic hardware like Intel's Loihi. I explain this choice carefully in §2.4, because it is *not* the choice the strongest paper recommends, and I want to be honest about that tension.

**IF — Integrate-and-Fire.** LIF with *no* leak. I mention it because one key paper argues IF is better on digital hardware — but since the shipping hardware implements LIF, I use LIF and discuss the disagreement openly rather than quietly picking the option that flatters my results.

**T — timesteps.** An SNN processes each input over `T` discrete time slices, not in one shot. I use `T = 25`.

**N_spikes/syn — spikes per synapse per inference.** The field's main sparsity number: on average, how many spikes cross each connection during one classification. Low is good for SNNs. It's a *count*, not a rate.

**N_syn / N_neur — synapses per neuron.** A structural property of the architecture. It decides whether a leaky neuron is affordable at all (§2.5). Bigger is friendlier to spiking.

**Process node (nm — nanometre).** A label for a chip-manufacturing generation. "45 nm" is silicon from about 2007–2008. The number once meant transistor size but for years has mostly been a marketing name for a technology generation. For scale: TSMC started volume production of its **2 nm-class (N2)** node in late 2025, so the energy numbers I use describe silicon roughly *twenty generations old*. That's why I quote **ratios** (which survive a node change) rather than absolute joules (which don't).

**SRAM / DRAM — on-chip / off-chip memory.** SRAM sits next to the compute and is fast; DRAM is bigger, farther, and far more energy-expensive to reach. The single most important hardware fact in this whole notebook: **reaching memory costs far more than doing the arithmetic.**

**CIM / IMC — Compute-in-Memory / In-Memory Computing.** A different hardware style that does the multiply *inside* the memory array, so weights never have to be fetched. It changes this entire analysis, and it's where commercial hardware is actually heading — so it gets its own treatment in §4 and §6-CIM, and it's central to my "what a fair next step looks like" discussion.


## Which number format is used where, and why (please read before comparing anything)

This confused me for a while, and it matters, because a lot of the "SNNs are efficient" literature quietly compares things at *different* precisions, which isn't fair.

Two separate things are going on:

**1. Operation counts are precision-independent.** A MAC is a MAC and an AC is an AC no matter the number format. So every MAC/AC/spike count in this notebook is exact and format-free.

**2. Energy-per-operation depends heavily on the format.** From Horowitz (2014) **[READ]** — the primary source everyone eventually cites — here are the actual measured costs at 45 nm:

| Operation | 8-bit integer | 32-bit integer | 16-bit float | 32-bit float |
|---|---|---|---|---|
| Add | 0.03 pJ | 0.1 pJ | 0.4 pJ | 0.9 pJ |
| Multiply | 0.2 pJ | 3.1 pJ | 1.1 pJ | 3.7 pJ |

A 32-bit-float MAC (≈ 3.7 + 0.9 = 4.6 pJ) costs about **20× an 8-bit-integer MAC** (≈ 0.2 + 0.03 = 0.23 pJ). That 20× has **nothing to do with spiking** — it's just precision. So if you compare a spiking network against a 32-bit-float ANN, you hand the ANN a huge precision penalty and call the difference an "SNN win." That's the trap §2.6 is about.

**What I actually use, and why:**

| Where | Format assumed | Source | Why |
|---|---|---|---|
| **Model (a), op-count** | FP32 *and* INT32, side by side | Chen et al. **[READ]** / Datta et al. **[READ]** | The two constant sets disagree ~6× in ratio; showing both makes a constant-dependent conclusion visible |
| **Model (b), memory-inclusive** | **INT8**, normalised so 1 MAC = 1 | Dampfhoffer et al. **[READ]** | INT8 is standard for real edge inference, and this is the model that counts memory (the ~99% op-count ignores) |
| **The "fair ANN baseline"** | low-bit INT (~`ceil(log₂(T+1))` bits) | Yan et al. **[READ]** | An SNN over `T` steps resolves only ~`log₂(T+1)` bits, so its honest comparator is a *low-bit* ANN, not FP32 (§2.6) |

**Every layer within a given energy model uses the same format** — I do not mix FP32 in one layer and INT8 in another. The point of reporting two whole models is to bracket the answer, not to cherry-pick per layer. Bottom line: **INT8 is my main assumption (Model b); I show FP32/INT32 op-counts alongside so you can see how much the format choice alone moves things.**


## 1 · Why hybrid networks exist

**ANNs** are accurate and easy to train, but every layer does dense MACs on every input, whether or not that input carries information.

**SNNs** talk in binary spikes, and a silent neuron is free on event-driven hardware. But they carry two burdens:

1. **Training is harder.** The spike function is a step; its derivative is zero almost everywhere and infinite at the threshold — useless for gradient descent. The standard fix is the **surrogate gradient**: the true step on the forward pass, a smooth differentiable bump on the backward pass. `snntorch` supplies these; I use a fast-sigmoid surrogate.
2. **The timestep penalty.** An SNN runs `T` timesteps per inference. If a layer is dense (real-valued inputs), running it `T` times is `T×` the work.

A **hybrid** keeps each half where it's strongest: dense ANN layers where the signal is busy, sparse SNN layers where it's quiet. The strongest hardware-aware paper I found reaches exactly this conclusion — hybrid ANN-SNN architectures, using event-based processing in high-sparsity layers and parallel ANN processing elsewhere, are a promising path to further energy savings (Dampfhoffer et al., 2023, [HAL open copy](https://cea.hal.science/cea-03852141v1) **[READ]**).

**The concrete question I make testable:** *which* layers should be which, and how much does the boundary placement actually matter once you count memory honestly?


---
## 2.0 · The hardware both sides are costed on *(read this before any number)*

> Every energy figure in this notebook comes from comparing two networks **on the same
> imaginary chip**. Which chip that is decides what the numbers mean, so here it is, explicitly.

### The machine

Both the ANN and the SNN are costed on a **spatially-expanded digital accelerator**: all weights
and neuron states live in on-chip SRAM, there is no off-chip DRAM, and compute and memory are
separate boxes (a von Neumann design). One process node, one number format, both sides.

This is the source paper's deliberate choice, and it says so plainly:

> *"to fairly compare ANNs and SNNs, we consider that both use a spatially expanded architecture
> in order to use the same memory energy values."* — Dampfhoffer et al. §II-A **[READ]**

### What each side is — and is not

| | What it actually is |
|---|---|
| **ANN side** | A *hypothetical* digital accelerator. Two versions are modelled: **naive** (no data reuse, §2.2) and **ideal reuse** (Setup cell 3b). |
| **SNN side** | A *generic* event-driven digital architecture, constructed by the paper. |
| **A GPU** | **Never modelled.** GPUs appear once in the paper — in the introduction, as the reason specialised hardware exists at all. No GPU number is in any equation. |
| **Loihi / TrueNorth / Akida** | **Never modelled.** Cited as real-world context. The paper says outright it uses *"a more general event-based architecture"* **instead of** taking an existing neuromorphic accelerator. |

Two consequences I want stated before anyone reads a result:

1. **My numbers cannot be compared against published Loihi measurements.** Those measure a
   specific chip, usually against an ANN on a GPU — two variables changing at once. Mine hold the
   hardware fixed and vary only the network. Different questions; §2A discusses the tension.
2. **Nothing here is a hardware measurement.** Operation counts are exact; joules are modelled.

### Is this apples to apples? Two answers, and both matter

**On hardware — yes, deliberately.** Same architecture, same memory, same node, same precision,
same constants for both sides. That is the entire point of the method.

**On design effort — no, and this is the honest gap.** The SNN at `16.33` is close to its *best*
case: no spike routing, no buffering, perfect event skipping. The naive ANN at `22.6` is a
deliberately *bad* design that refetches everything. I am comparing a well-designed SNN against a
badly-designed ANN. **That is why Setup 3b adds a second, well-designed ANN**, and why every
headline number in §6.4 is quoted under both.

### Will newer hardware break this?

The useful split is not "old vs new silicon" but **ratio-preserving vs ratio-breaking**.

| Advance | Who it helps | Threatens these conclusions? |
|---|---|---|
| Node shrink (45 nm → 2 nm) | both, proportionally | **No.** Exactly why I quote ratios, never absolute joules. |
| Lower precision (INT8 → INT4) | both, proportionally | **No.** Compute and memory shrink together. |
| Better dataflow / data reuse | **ANN only** | **Yes** — quantified in Setup 3b. |
| Pruned / zero-skipping accelerators | **ANN only** | **Yes**, and not modelled here. Direction: against the SNN. |
| Sparser SNN training algorithms | **SNN only** | **Yes**, in the SNN's favour. |
| 3D stacking, HBM, better DRAM I/O | neither | **No** — this model has no DRAM. |
| **Compute-in-memory** | abolishes the fetch for **both** | **Inverts the mechanism.** Different lens needed → **Appendix A**. |

> **The one-sentence version.** Anything that scales compute and memory *together* leaves every
> ratio in this notebook intact, so the project extends to newer nodes unchanged. Anything that
> hands **one** side a capability the other structurally cannot use — data reuse for the ANN,
> event-skipping for the SNN — has to be re-derived, and I re-derive the biggest one in Setup 3b.
> Compute-in-memory is the single case that *inverts* the argument rather than scaling it, which
> is why it gets an appendix rather than a footnote.

## 2 · The mathematics — and the correction I almost missed

Almost every SNN paper estimates energy by counting arithmetic:

$$E \;=\; N_{\text{MAC}} \cdot E_{\text{MAC}} \;+\; N_{\text{AC}} \cdot E_{\text{AC}}$$

then notes ACs are cheaper than MACs, so SNNs "must" win. **On digital hardware this counts only about 1% of the real energy.** The other ~99% is memory access — fetching weights and neuron states. This is the correction the whole notebook is built around (Dampfhoffer et al. **[READ]**).

> **This isn't a fringe claim — it's Horowitz's own point.** Mark Horowitz's 2014 ISSCC plenary *"Computing's Energy Problem (and what we can do about it)"* (ISSCC 2014, Session 1.1; I read the PDF, which is provided alongside this project) **[READ]** — the origin of essentially every energy constant in this field — has a section literally titled *"Don't Forget the Memory Energy."* He states a DRAM access (1.3–2.6 nJ) costs "a couple of orders-of-magnitude" more than a functional operation (~10 pJ), and that over 50% of a processor's energy goes to caches and register files. So "memory dominates" is the primary source's headline, not my reinterpretation.

### 2.1 · The constant that reorganises everything

From Dampfhoffer's Table I (45 nm, INT8, every cost normalised so one MAC = 1) **[READ]**:

$$E_{\text{AC}} = 0.13, \qquad E_{\text{MAC}} = 1, \qquad E_{R/W} = 5.4$$

**A single SRAM read or write costs 5.4 MACs.** Arithmetic is noise next to memory traffic.

> **Honest sourcing note — I had to correct myself here.** After actually reading Horowitz, I found the widely-quoted pair "E_MAC = 4.6 pJ, E_AC = 0.9 pJ (FP32, 45 nm)" is **not written in Horowitz's paper**. His table gives 32-bit FADD = 0.9 pJ (the AC) and 32-bit FMULT = 3.7 pJ, which *sum* to 4.6 for a MAC — but he never writes "4.6" and never labels a "MAC." So the FP32 pair is a *reasonable sum by later authors* (Chen et al.), not a Horowitz quote. His INT32 pair (3.1 + 0.1 = 3.2 MAC, 0.1 AC) **is** directly his. I now tag Horowitz **[READ]** and say plainly: the FP32 MAC number is an inference from two of his numbers, not a direct quote. It changes no conclusion here — everything load-bearing is a *ratio* in Dampfhoffer's normalised INT8 units — but the attribution had to be fixed.

### 2.2 · Cost per synaptic operation, by hand

Count *every* memory access a synapse actually incurs, in units of one MAC:

**Naive ANN, per synapse** — read input activation, read weight, read partial sum, write partial sum, one MAC:

$$E_{\text{ANN}} = 4(5.4) + 1 = \mathbf{22.6}$$

**SNN (LIF neuron), per spike** — read weight, read state, write state, one AC. *No input-activation read: a spike is its own address — the hardware already knows which synapse fired.*

$$E_{\text{SNN}} = 3(5.4) + 0.13 = \mathbf{16.33}$$

**The SNN's real structural advantage is one fewer memory access — not the MAC→AC swap.** The memory saving is 5.4; the arithmetic saving is only $1 - 0.13 = 0.87$. The memory effect is **6.2× larger** than the arithmetic effect everyone talks about.

### 2.3 · Break-even: SNNs are not automatically cheaper

Set the two equal. An SNN beats a naive ANN only if

$$N_{\text{spikes/syn}} < \frac{22.6}{16.33} = \mathbf{1.38} \text{ spikes per synapse.}$$

Above ~1.38 spikes per synapse, the "efficient" SNN is *more* expensive than the dumb ANN. And 1.38 is the *easy* target — it assumes the worst possible ANN. Against a good ANN accelerator that reuses data, the target tightens sharply (Dampfhoffer Table III **[READ]**):

| ANN implementation | SNN must fire fewer than … | From which model? |
|---|---|---|
| Naive baseline (worst case) | 1.38 spikes/syn | **45 nm / 8-bit** (paper §II) — the one we use |
| Ideal data reuse | 0.28 | 65 nm / 16-bit (paper §III-A) |
| Ideal reuse + activation sparsity (best case) | **0.15** | 65 nm / 16-bit (paper §III-A) |
| Eyeriss v1 (a real chip) | 0.42 | 65 nm / 16-bit (paper §III-B1) |
| Eyeriss v2 | 0.37 | 65 nm / 16-bit (paper §III-B2) |

> **⚠️ Do not try to derive the last four from the arithmetic above — you will fail.** Only **1.38** comes from `22.6 ÷ 16.33`. The other four come from a *different part of the paper* that switches to a **different technology node and precision** (65 nm, 16-bit: `E_AC = 0.06`, `E_R/W = 6`, plus a register-file cost of 1) **and** adds machinery this notebook does not model at all: per-layer Reuse Factors, local register-file energy, and a 58% zero-activation rate (their Eqs. 11–13).
>
> **What this means for us.** Our whole notebook uses the 45 nm / 8-bit model, so our numbers are internally consistent. But the 0.42 / 0.37 Eyeriss figures are **not** on the same scale as anything we compute — quote them as *"the paper reports that against real accelerators the target tightens to ~0.4, at a different node"*, never as though our model produced them.

**Data reuse** is the term every op-count model omits: a good ANN reads a weight once and reuses it across hundreds of MACs locally (reuse factors $10^2$–$10^3$); an event-driven SNN *can't*, because spikes arrive in an unpredictable order. That one effect tightens the SNN's required sparsity ~9×.

> ### ⚠️ Two corrections that matter for *this* project
>
> **(a) 1.38 is the *IF* threshold. We use LIF.** The paper derives 1.38 for the **IF + instantaneous synapse** model — the one with **no leak** — and says it holds *"regardless of the network topology"*. That topology-independence is exactly *because* IF has no per-timestep term, so `N_syn` cancels cleanly. For the leaky models the paper is explicit: *"To compare the other SNN models, which requires an update at each timestep, we must take into account the network topology and the temporal resolution."*
>
> Since this project uses **LIF**, the honest threshold carries the leak:
>
> $$N_{\text{spikes/syn}} \;<\; T \cdot \frac{22.6 - 11.8 \,/\, (N_{\text{syn}}/N_{\text{neur}})}{16.33}$$
>
> At our ratios that works out to 34.57 (SHD-700) and 34.25 (40-channel) against the simple `T × 1.38 = 34.5` — a 0.2–0.7% difference, so the simple form is safe **here**. It would not be at a low ratio: at `N_syn/N_neur ≈ 10` the true threshold is 32.9, nearly 5% lower.
>
> ### ⚠️ (b) 1.38 assumes the ANN runs **once**
>
> This tripped me up, and getting it right changes how every result below is read.
>
> The 1.38 figure compares **one** ANN synaptic operation against **one** SNN spike. That is the right comparison for a **static** input like an image, where the ANN makes a single forward pass.
>
> But my ANN processes a **time series** — a spectrogram with `T = 25` frames — and it runs over its synapses **once per frame**. So per inference:
>
> $$E_{\text{ANN}} = N_{\text{syn}} \times T \times 22.6 \qquad\qquad E_{\text{SNN}} = N_{\text{syn}} \times N_{\text{spikes/syn}} \times 16.33$$
>
> Setting them equal gives the threshold that actually applies here:
>
> $$N_{\text{spikes/syn}} < T \times 1.38 = 25 \times 1.38 = \mathbf{34.5} \text{ spikes per synapse.}$$
>
> **This is our extension, not something the paper says.** Dampfhoffer's Eq. 2 has no `T` in it — their ANN is an *image* classifier making a single forward pass. Our ANN classifies a 25-frame spectrogram, so it runs over its synapses 25 times. The algebra below is a straightforward consequence, but **do not attribute `34.5` to the paper** — attribute it to us, applying their model to a time-series task.
>
> **So on this task the SNN has 25× more headroom than the raw 1.38 suggests** — because the ANN it is competing against is itself paying a 25× timestep tax. Quoting 1.38 against a `T`-pass ANN would understate the SNN's budget by a factor of `T`.
>
> **Two rules that follow, and I apply both below:**
>
> 1. **Compare `N_spikes/syn` against `T × 1.38`, not 1.38**, whenever the ANN baseline processes a time axis frame-by-frame.
> 2. **This threshold is only valid at equal topology — and the paper assumes it throughout.** Every comparison in Dampfhoffer is between an ANN and *its own* spiking counterpart: the same VGG16, the same AlexNet, converted. They never compare a wide network against a narrow one. Our Section 2 does exactly that (SHD's 700 channels vs HD's 40), which is a genuine departure from the source method and is why the matched-width control in Section 5 is the one that licenses a verdict. It cancels `N_syn` from both sides, so it says nothing about two networks with *different* synapse counts. If the input widths differ (SHD's 700 channels vs HD's 40 — a 12× synapse difference), you must compare the full energies, not the spike rate. That is exactly why Section 5 exists, and why cross-dataset claims come only from there.

### 2.4 · Why I use LIF — and why that's the honest choice even though one paper prefers IF

This is the subtlest decision in the project, so I'll walk through it rather than hide it.

A **leaky** neuron must update its membrane potential *every neuron, every timestep*, spike or not — a read-modify-write with a multiply:

$$E_{\text{LIF leak}} = 2(5.4) + 1 = \mathbf{11.8} \text{ per neuron per timestep.}$$

Dampfhoffer found no accuracy or sparsity benefit from the leak on *their* image tasks, so they concluded **IF (no leak) is the better digital choice [READ]**. Reading that, my first instinct was to use IF — it would make my SNNs look cheaper.

**Here's why using IF would have been slightly dishonest, and why I use LIF instead:**

- **The commercial hardware implements LIF, not IF.** Intel's Loihi (both generations) has the current-based LIF (CUBA-LIF) neuron as its *built-in hardware primitive* — verified from multiple implementation papers ([Loihi klinokinesis paper, Eq. for LIF](https://arxiv.org/abs/2105.01358) **[READ]**; [Brian2Loihi emulator](https://www.frontiersin.org/journals/neuroinformatics/articles/10.3389/fninf.2022.1015624/full) **[READ]**). On Loihi you get IF only as a *special case* of LIF by setting the decay to zero. If I want the project to reflect hardware that actually ships, LIF is the honest default.
- **Dampfhoffer's "IF is better" is conditional**, not a general law. It's their conclusion, for their image tasks, under a digital von-Neumann accounting. My data is *temporal audio* (SHD), where the leak gives a neuron short-term memory/forgetting — a property that plausibly *helps* on time-series and is exactly what leaky dynamics are for. Whether the leak earns its cost here is something to *measure*, not assume.
- **Choosing IF would have been choosing the option that flatters SNN energy.** My working rule for this project: when a choice makes my preferred side look better, be extra careful. LIF is the more conservative, more hardware-realistic choice.

> **Note to self / to the reader:** using LIF means my SNNs pay the full leak tax `N_neur × T × 11.8` in every section. That makes the "put the readout in ANN-land so it has no membrane" trick (Section 3) *more* visible, and makes the hostile-ratio warning (Section 5) sharper. I think that's a feature: it's the realistic cost, and it's what the boundary-placement question is really about.

### 2.5 · The ratio that decides whether the leak is even affordable

The leak burden scales as

$$\text{leak burden} \;\propto\; \frac{N_{\text{neur}} \cdot T}{N_{\text{syn}} \cdot N_{\text{spikes/syn}}} \;=\; \frac{T}{\left(N_{\text{syn}}/N_{\text{neur}}\right)\cdot N_{\text{spikes/syn}}}$$

The smaller the **synapses-per-neuron** ratio, the more the leak dominates. Verified arithmetic for the architectures I use (`hidden = 128`, `T = 25`, ~0.5 spikes/syn):

| Config | N_syn | N_neur | ratio | Predicted leak share (at 0.5 spikes/syn) | **Measured leak share** |
|---|---|---|---|---|---|
| Tic-tac-toe 9→32→2 | 352 | 34 | **10.4** | ruinous | *(not run)* |
| HD 40 mel →128→20 | 7,680 | 148 | 51.9 | ~41% | **1.3% – 9.2%** |
| SHD 140→128→20 | 20,480 | 148 | 138.4 | ~21% | *(not run)* |
| **SHD 700→128→20** | 92,160 | 148 | **622.7** | ~5.5% | **0.5%** |

> **Correction after measuring (v10).** The "predicted" column assumed **0.5 spikes/synapse**. The real networks fire far more than that — **5.5 spikes/syn** at SHD-700 and **11.2** at SHD-40 (see §2.3 on why that is still well inside budget). More spikes means more synaptic traffic, which makes the leak a *smaller* slice of a bigger pie. So the predictions were **~10× too pessimistic** about the leak.
>
> **The mechanism still holds, directionally.** Drop the ratio 12× (623 → 52) and the leak's share rises ~5.7× (0.5% → 3.0%), exactly the direction §2.5 predicts. But the *magnitudes* in the original table were wrong because the assumed spike rate was wrong.
>
> **What this means in practice:** on these architectures the LIF leak is a rounding error, not a design constraint — synaptic traffic dominates everywhere. The leak only becomes decisive at genuinely small `N_syn/N_neur` (the tic-tac-toe regime), which is why that task was dropped. **Assume nothing about the leak; read the `lif_steps` column.**

An earlier version of this project used a tic-tac-toe task; at ratio ~10 the leak is ruinous, for reasons unrelated to the hybrid question, so I dropped it. **This is also why I keep SHD at its native 700 channels** — downsampling to 40 or 140 (common for speed) quietly wrecks the ratio and changes the conclusion. Section 5 runs the downsampled control on purpose, so the cost is visible instead of hidden.

### 2.6 · The fairness problem, explained slowly (this one took me a while)

Here's the cleanest way I can explain it.

An SNN runs for `T` timesteps. At each step an output neuron either spikes (1) or doesn't (0). Over the whole inference its total spike count can be any whole number from 0 to `T` — that's **`T + 1` possible distinct values**. A signal with `T+1` levels carries about **`log₂(T+1)` bits** of information. So an SNN's *effective output precision* is only about `log₂(T+1)` bits.

Concretely: at `T = 7`, that's `log₂(8) = 3 bits`. A 3-bit machine.

Now the unfairness: nearly every paper showing "SNN beats ANN on energy" compares the SNN against a **32-bit floating-point** ANN. But the SNN is effectively a ~3-bit machine. Comparing a 3-bit spiking network against a 32-bit-float ANN stacks the deck — you charge the ANN for 32 bits of precision the SNN never provides. And from the format table above, an FP32 MAC costs ~20× an INT8 MAC purely from precision, *nothing to do with spiking*.

**The fair comparison** pits the SNN against an ANN quantised to the *same* effective precision — roughly a `ceil(log₂(T+1))`-bit integer ANN (Yan et al., [arXiv:2409.08290](https://arxiv.org/abs/2409.08290) **[READ]**). Against *that* baseline, much of the SNN's apparent advantage shrinks, because low-bit integer ANNs are already very cheap.

**What I do about it:** I report both energy models precisely so this assumption can't hide. When you read "the SNN is X× cheaper," always ask: *cheaper than an ANN at what precision?* If the answer is FP32, discount it.

### 2.7 · A "pure SNN" on spectrogram audio is secretly a hybrid — and that's the whole reason I use two datasets

This point motivates my entire experimental design, so let me make it concrete for this project.

If the input is real-valued numbers — a mel-spectrogram, an image, an analog sensor reading — the very first layer *must multiply* (MAC), because there's nothing binary to accumulate. You can't AC the number 3.7; you can only MAC it (Datta et al., [arXiv:2107.12374](https://arxiv.org/abs/2107.12374), Eq. 25 **[READ]**).

So when someone says "I trained a pure SNN on audio spectrograms," their first layer is *actually a dense ANN layer in disguise* — and if they don't count it as MACs, their energy numbers are wrong in the SNN's favour.

The only way to get a *genuinely* spiking first layer is to feed data that is **already spikes**. That's exactly what SHD gives me: its cochlea model turns audio into spike events, so SHD's first layer can honestly be ACs.

**This is why my project pairs two datasets of the same spoken digits:**

- **HD** (Heidelberg Digits, raw audio → mel-spectrogram) → real-valued input → first layer is **forced MACs**. The honest "audio arrives as numbers" case.
- **SHD** (Spiking Heidelberg Digits, cochlea spikes) → binary input → first layer is genuine **ACs**. The honest "audio arrives as spikes" case.

Without this pairing, any "SNNs are efficient on audio" claim is quietly smuggling in a dense first layer and not paying for it. **My whole comparison is built to make that hidden cost visible.**

> **Matching caveat I keep flagging:** the cochlea model that made SHD's spikes, and the FFT that made HD's spectrogram, both cost real energy that *no* model here counts — so even my honest comparison starts one step too late. I state this wherever it matters.


---
### 2.3b · The comparison ledger — a rule I apply to every ratio below

Before quoting any "**X× better**", state which of **four axes** are matched:

| Axis | Status in this notebook |
|---|---|
| **Hardware** | Always matched — §2.0. Never a confound here. |
| **Dataset** | Matched *within* Sections 1–4 and *within* Section 7; **not** across them. |
| **Network width** | Matched only in Section 5, and in same-dataset pairs. |
| **Accuracy** | Matched almost nowhere — this is the binding constraint. |

Three verdicts follow:

- **All four matched** → a clean result. In this notebook only **S4a/S4b** and **V4/V4b** qualify.
- **The cheaper model is also equal-or-better on accuracy** → a **dominance** result. No
  iso-accuracy argument is needed, because there is no tradeoff to adjudicate. **S2/S3**,
  **V2/V3** and the **S5 pair** qualify.
- **Anything else** → quote the confound in the same sentence as the number.

§6.4 prints this ledger automatically from the measured results, so it cannot quietly go stale.

---
## 2A · Is the paper this project leans on actually accepted? (a deliberate check for blind spots)

Almost every quantitative claim in this notebook traces back to one paper — **Dampfhoffer et al. (2023)**. That is a single point of failure. If its method were disputed, this project would inherit the dispute silently. So before building on it, I checked whether its findings are consensus or contested. Here is what I found, with links.

### The verdict, up front

**The core methodological claim is not just accepted — it has been independently reproduced by several other groups, in peer-reviewed venues.** I found **no published refutation**. What I did find is a real and important tension about *scope*, described below, which the project must state rather than hide.

### Who independently reached the same conclusion

| Source | Venue | What they concluded |
|---|---|---|
| [Dampfhoffer et al. (2023)](https://cea.hal.science/cea-03852141v1) **[READ]** | IEEE TETCI (peer-reviewed) | Memory access dominates; sparsity is the real lever, not MAC→AC; hybrids are promising |
| [Ottati et al. (2023)](https://arxiv.org/abs/2306.15749) **[READ]** | IEEE JETCAS (peer-reviewed) | "Spikes are efficient **only** when exploiting their sparsity"; and explicitly: *"a mixed approach takes advantage of both"* — an independent endorsement of hybrids |
| [Yan, Bai & Wong (2024)](https://arxiv.org/abs/2409.08290) **[READ]** | arXiv | Counting only additions "can lead to a misleading perception of efficiency"; SNNs need **strict** limits on both timestep count `T` and sparsity |
| [*Is Conventional SNN Really Efficient?* (2023)](https://arxiv.org/abs/2311.10802) **[READ]** | arXiv | Comparisons are "often skewed… lacking fairness towards ANNs"; proposes a "bit budget" view |
| [*Rethinking the Energy Efficiency of SNNs and ANNs* (2026)](https://dl.acm.org/doi/10.1145/3822176) **[READ]** | ACM TACO (peer-reviewed) | Current SNN designs "primarily focus on synaptic operations… often overlooking the impact of memory access" |

Five groups, three of them in peer-reviewed IEEE/ACM journals, converging on the same methodological point across four years. For a student project, that is about as safe a foundation as this field offers.

### The genuine tension — and it does cut against us

Here is the part that would be dishonest to omit. **Studies that measure real neuromorphic hardware frequently report SNNs winning by large margins**, which sits awkwardly beside the analytical papers above. Examples I read:

- A human-activity-recognition study measured **SNN on Loihi at 0.15 mJ vs ANN on a RISC-V GAP8 at 0.41 mJ** — the SNN ~2.7× better, at essentially matched accuracy (87.5% vs 88.1%).
- An event-based optical-flow study with hardware-in-the-loop found the **SNN used 75% of the ANN's energy**.
- Robotics and SLAM studies on Loihi report 20×–100× advantages over CPU/GPU baselines.

**How to reconcile this honestly.** Two things explain most of the gap, and both are worth saying aloud:

1. **Most measured comparisons change two variables at once.** They compare an *SNN on a neuromorphic chip* against an *ANN on a GPU or CPU*. That measures the ASIC-vs-general-purpose-processor gap at least as much as it measures spiking. The better studies notice this — one space-applications paper explicitly re-ran the ANN *on Loihi too* "to eliminate the effects of the hardware."
2. **The analytical papers and the measured papers are answering different questions.** Analytical work asks *"at equal hardware, is spiking intrinsically cheaper?"* Measurement asks *"is this deployed spiking system cheaper than this deployed conventional system?"* Both are legitimate; they are not the same question, and the answers differ.

Notably, the measured results agree with Dampfhoffer on the *mechanism* even when they disagree on the verdict: the optical-flow authors attribute their SNN's advantage to **lower spike density requiring fewer memory accesses for neuron states** — which is precisely Dampfhoffer's claim that memory traffic, not arithmetic, is what matters.

### Known limitations of Dampfhoffer's study that this project inherits

Being specific about these is more useful than a generic disclaimer:

- **It is analytical, not measured.** No chip was run. So is this notebook. Both are estimates.
- **Its ANN baseline may not be the strongest possible.** A later study observes that although zero-activation skipping was considered, *"there is no work yet comparing SNNs with sparsified ANNs"* — i.e. a heavily pruned/sparsified ANN might do even better than the baseline used, which would tighten the SNN's target further. This cuts **against** the SNN, so our conclusions are conservative in that direction.
- **Compute-in-memory is explicitly out of scope** — and on CIM the argument partly inverts (see the CIM discussion later).
- **The constants are ~45 nm silicon (circa 2008).** Quote ratios, not absolute joules.
- **Fair comparison remains genuinely unsolved.** A 2025 survey of this exact question concludes that "fair comparison of SNN and ANN energy consumption remains a challenge," with hardware-aware and hardware-agnostic methods giving different answers.

### What this means for how you should present the project

State it like this, and you will be on solid ground:

> *"The accounting method I use is the mainstream hardware-aware one, independently corroborated by several groups in peer-reviewed venues, and I found no published refutation. It is an analytical estimate, not a hardware measurement — and measured neuromorphic studies often report larger SNN advantages, mostly because they compare a neuromorphic chip against a GPU, which changes the hardware and the network at the same time. My numbers are therefore best read as 'at equal digital hardware', and as ratios rather than absolute joules."*

That framing is both accurate and defensible under questioning — including the obvious challenge, *"but I read that neuromorphic chips are 100× more efficient."*

## 3 · How hybrid ANN-SNN networks are being studied today

Before claiming any ground, I wanted to know who my neighbours are. Hybrid ANN-SNN is an active, crowded subfield, not an open frontier. These are sources I opened this session, tagged honestly and linked:

- **Dampfhoffer, Mesquida, Valentian & Anghel (2023), IEEE TETCI** — [HAL open copy](https://cea.hal.science/cea-03852141v1) **[READ]**. The hardware-aware energy model my whole notebook rests on. Concludes: the SNN advantage comes mainly from *spike sparsity*, not MAC→AC; hybrids — ANN where activity is high, SNN where it's sparse — are the promising path. Their worked ResNet-34 example: network-wide the ANN is 6.3× better, but the sparse last 10 layers are 2.1× better as SNN, and the hybrid (first 22 ANN, last 12 SNN) is 1.2× better than the full ANN.
- **Seekings et al. (2024)** — [arXiv:2407.08704](https://arxiv.org/abs/2407.08704) **[READ]**. A layer-by-layer boundary sweep on event-vision, **measured on real Loihi + Jetson hardware** (rare — most work is analytical). Two findings I echo: adding spiking layers lowers energy, but making *every* conv layer spiking blew the parameter count from ~224k to ~3M because the accumulator expanded the channel dimension — a real trap.
- **Luu, Luu, Nam & Thang (2025)** — [arXiv:2509.24411](https://arxiv.org/abs/2509.24411) **[ABSTRACT]**. A *layer-wise* encode/decode hybrid that interleaves SNN blocks inside an ANN pipeline via a differentiable bit-plane spike encoding, so gradients flow across the seam. The "tightly-interleaved" end of the design space, versus my "one clean boundary" end.
- **Hybrid SNN-ANN DenseNet (2021)** — [arXiv:2112.03423](https://arxiv.org/abs/2112.03423) **[ABSTRACT]**. SNN backbone → accumulator → ANN head, the same SNN-front → ANN-back pattern my Section 3 measures.
- **Louis, Pegatoquet, Miramond & Girard (2025), IEEE** **[ABSTRACT]**. CIFAR-10 + Google Speech Commands, five architectures, up to **2.45× energy reduction** (energy *estimated*). This is where my "realistic prize is 1.2–2.45×, not orders of magnitude" framing comes from.

**Honest positioning.** Several groups have swept the ANN/SNN boundary; at least one (Seekings) did it on real hardware. My searches this session did **not** surface anyone running this *exact* boundary sweep on the SHD audio dataset with a matched HD-spectrogram baseline — but "my searches didn't find it" is a much weaker claim than "nobody has done it," and I only make the weaker one. My value here is a clean, fully-audited replication a student can run and defend, not a land-grab.


## 4 · Neuromorphic and compute-in-memory hardware: where it actually is today

My notebook runs on a GPU and *models* neuromorphic energy — it touches no special hardware. But the whole motivation assumes such hardware exists, so here's the honest commercial picture as of mid-2026, from sources I opened this session.

### Spiking (neuromorphic) hardware

**Intel Loihi 2 / Hala Point** — [Intel newsroom](https://newsroom.intel.com/artificial-intelligence/intel-builds-worlds-largest-neuromorphic-system-to-enable-more-sustainable-ai) **[READ]**. The largest neuromorphic system built to date: 1,152 Loihi 2 chips, 1.15 billion neurons, ~2,600 W, in a chassis the size of a microwave, deployed at **Sandia National Laboratories**. Intel is explicit that it's a **research prototype**, "not commercially available." One applied pilot: **Ericsson Research** using Loihi 2 for telecom-infrastructure efficiency. Two details matter for my project: (1) **each Loihi core co-locates SRAM with compute** for neuron state and weights — so Loihi is itself a form of compute-in-memory, and neurons "communicate directly rather than through memory." This has been true since Loihi 1; it's the neuromorphic design principle, not a 2024 change. (2) **Loihi's built-in neuron is LIF** (CUBA-LIF) — exactly why I use LIF (§2.4).

**BrainChip Akida** — [company site](https://brainchip.com/) + trade press **[READ]**. The clearest example of *commercial* neuromorphic silicon: a fully-digital, event-based edge processor sold as chips and licensable IP. As of late 2025 / early 2026 the traction is real but still mostly **evaluation and pilot** stage — a strategic agreement with Parsons for defense edge-AI ([announcement](https://investor.brainchip.com/brainchip-and-parsons-sign-strategic-agreement-to-accelerate-edge-ai-defense-systems/) **[READ]**), an AKD1500 co-processor for wearables/sensors, and a $25M raise ahead of CES 2026. Notably, their newer generation leans on **temporal / state-space (TENN) models** for streaming audio and video — i.e. leaning *into* temporal dynamics, the opposite of throwing the leak away.

**Takeaway.** Spiking hardware in 2026 is *real* but sits between research-prototype (Loihi) and early-commercial-edge (Akida). It is not yet a mainstream datacenter alternative to GPUs — and it implements LIF.

*(The compute-in-memory half of this section has moved to **Appendix A**, because CIM is the one hardware change that inverts my argument rather than scaling it — see §2.0. It is out of scope here, not unimportant.)*

---
## 4b · Why SNNs are still being funded *(the question §2A raises and doesn't answer)*

§2A showed that measured hardware studies often report SNNs winning while analytical studies like
mine often do not. §4 showed the commercial hardware is still early. A fair question follows:
**if the accounting is this unflattering, why is anyone still building these?**

I think there are five real answers — and the uncomfortable thing about them is that **four of the
five are invisible to my model.** Better to say that out loud than let my energy number stand as
the whole story.

**1 · Latency, which this notebook does not measure at all.** The source paper is explicit about
its scope: it focuses *"on energy consumption rather than area, latency and throughput"* **[READ]**.
An event-driven network can begin producing an answer as soon as enough evidence has arrived,
rather than after a fixed number of frames — the basis of time-to-first-spike coding. For a
wake-word detector or a collision-avoidance sensor, latency may matter more than joules. My model
is blind to this by construction.

**2 · Sensor co-design.** If the sensor is *already* event-based — an event camera, a silicon
cochlea — then an ANN must first convert those events into dense frames before it can do anything.
That conversion costs real energy and **no model here counts it.** It is the exact mirror image of
the uncounted-cochlea caveat I keep applying *against* the SNN, and it falls on the ANN. Both are
uncounted; I should not quote one asymmetry without the other.

**3 · Always-on duty cycling.** For a device that is listening but silent 99% of the time, static
(leakage) power and wake-up cost dominate the budget. My model is **dynamic energy only** — the
paper excludes static power explicitly, and so do I. A network whose neurons are genuinely *idle*
rather than merely computing zeros is a different proposition on that workload.

**4 · Sparse, bursty signals — and this one I actually measured.** N-MNIST fires **1.06** spikes
per synapse against SHD's **5.46**. Where the signal is genuinely sparse, the event-driven approach
does what it promises. That is not a hope; it is Section 7's data.

**5 · Compute-in-memory synergy.** On analog crossbars the ADC dominates the energy budget, and
binary activations permit a very low-precision ADC. A real advantage — but a *different* one from
the one this notebook measures. It lives in **Appendix A**.

> **The honest summary.** My notebook measures dynamic synaptic energy on a digital von Neumann
> accelerator, for a network that is always on, fed by a sensor whose cost nobody counts. Within
> that frame the SNN's case is narrow and contingent on sparsity. Four of the five reasons the
> field keeps investing sit **outside** that frame. *"My model says the SNN does not dominate"* and
> *"the SNN is not worth building"* are different claims, and only the first one is mine.

## 5 · What this notebook measures, and the two energy models

For each architecture below I report, per inference:

- **Test accuracy** (mean ± std over 3 seeds).
- **N_spikes/syn** per layer — the field's sparsity metric, measured from real spike traces, not assumed.
- **MAC and AC counts**, separated by whether the driving activation is binary (AC) or real-valued (MAC).
- **Membrane updates** (`N_neur × T`) — the LIF leak term that costs energy every timestep.
- **N_syn / N_neur** — the ratio from §2.5.
- **Energy under two independent models:**
  - **Model (a) — operation count only.** `E = MACs·E_MAC + ACs·E_AC`. What most SNN papers use. Reported under *two* published constant sets (FP32 and INT32) so a constant-dependent conclusion is visible.
  - **Model (b) — memory-inclusive (Dampfhoffer, INT8).** Charges every synapse its real memory traffic (§2.2), normalised to one MAC.

**My discipline throughout:** report both models. Where they agree, the ranking is robust to the choice of constants. **Where they disagree, that disagreement is itself the finding**, and the memory-inclusive model is the more defensible number on digital hardware.

**Operation and spike counts are exact measurements. Joules are modelled.** I keep those two words — *measured* and *modelled* — separate everywhere.

> **Runnable without any dataset.** Set `USE_SYNTHETIC = True` in the config and the whole counting/energy story runs on random tensors of the right shape. Accuracy is then meaningless (near chance), but every operation count and both energy models are exact and real. To reproduce accuracy you need the real datasets (SHD via `tonic`, HD via its [audio archive](https://compneuro.net/datasets/hd_audio.tar.gz)), which the loader cells fetch.


---
## Setup cell 1 — install dependencies (preserves Colab's existing NumPy)

Run once per session (~1 min on Colab). `snntorch` supplies the spiking neurons and surrogate gradients; `tonic` loads SHD; `torchaudio` decodes the HD audio.

> **Why this cell is careful about NumPy.** Colab already ships a *mutually consistent* stack (NumPy 1.26, pandas 2.2, scikit-learn 1.6, etc.). The one dependency that can break it is `tonic`, which pins `numpy<2.0.0` and can trigger pip to reshuffle NumPy mid-session — which corrupts the already-imported NumPy and later throws `No module named 'numpy.rec'` or `numpy.dtype size changed`. To avoid that, this cell installs `tonic` (and `snntorch`) **without touching NumPy/pandas/scikit-learn**: it adds only the few genuinely-missing small packages, then installs `tonic`/`snntorch` with `--no-deps`. So it should *not* require a restart. If you ever do see a `numpy.*` error anyway, it means the session's NumPy was already disturbed — just do **Runtime → Restart session** and run from the top; you won't need to reinstall.

In [ ]:
import sys, subprocess
def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=False)

# tonic's small deps that Colab may lack. h5py/scipy/librosa/tqdm are already on Colab,
# so we only add the three that typically aren't -- and none of these pull NumPy.
pip("importRosbag>=1.0.4", "pbr", "expelliarmus", "soundfile")

# Install snntorch and tonic WITHOUT dependency resolution, so pip cannot upgrade or
# downgrade the NumPy/pandas/sklearn that Colab already has working.
pip("--no-deps", "snntorch", "tonic")

# ---- verify the existing NumPy/pandas ABI is still intact in THIS process ----
_ok = True
try:
    import numpy as np, numpy.rec        # numpy.rec is exactly what breaks on a bad reshuffle
    import pandas
    _x = np.ones(2, dtype=np.float32)
    assert abs(float(_x.dot(_x)) - 2.0) < 1e-5
    print("numpy    ", np.__version__, "(healthy)")
    print("pandas   ", pandas.__version__)
except Exception as e:
    _ok = False
    print("*** NumPy/pandas disturbed in this session:", type(e).__name__, e)
    print("    FIX: Runtime -> Restart session, then run from the top (no reinstall needed).")

import torch
print("torch    ", torch.__version__)
try:
    import snntorch; print("snntorch ", snntorch.__version__)
except Exception as e: print("snntorch  MISSING:", e)
try:
    import tonic; print("tonic    ", tonic.__version__)
except Exception as e: print("tonic     (optional) not available:", e)
print("CUDA     ", torch.cuda.is_available())

if not _ok:
    print("\n>>> STOP: restart the runtime now, then re-run from the top. <<<")

---
## Setup cell 2 — configuration

Every knob lives here. Defaults are chosen deliberately and explained in the front matter:

- **`T = 25`** — same timesteps for both datasets, so the time axis is never a confound.
- **`N_MELS = 40`** — standard speech front end, gives HD a realistic baseline.
- **`SHD_CHANNELS = 700`** — *native*. Downsampling wrecks `N_syn/N_neur` (§2.5); Section 5 runs the 40-channel control on purpose.
- **`NEURON = "LIF"`** — I use the leaky neuron only, because it is the one commercial neuromorphic hardware (Intel Loihi) actually implements. §2.4 explains this choice in full, including why I *don't* use the no-leak IF neuron even though one paper prefers it.
- **`USE_SYNTHETIC`** — when `True`, every section runs on random tensors of the correct shape. Accuracy is then meaningless (it will hover near chance), but **all operation counts, spike-structure invariants, and both energy models are exact and real**. This lets the counting/energy narrative execute anywhere, even with no dataset egress. Set to `False` to reproduce the real accuracy numbers.

In [ ]:
import torch, numpy as np, random, os, json

CFG = dict(
    T            = 25,      # timesteps / frames (both datasets)
    N_MELS       = 40,      # HD mel bins
    SHD_CHANNELS = 700,     # native SHD resolution
    MATCHED_CH   = 40,      # Section 5 control
    HIDDEN       = 128,
    N_CLASSES    = 20,
    BATCH        = 128,
    EPOCHS       = 20,
    LR           = 2e-3,
    BETA         = 0.9,     # LIF membrane decay (per timestep)
    SEEDS        = [0, 1, 2],
    NEURON       = "LIF",   # LIF only: it is what commercial neuromorphic HW implements (see 2.4)
    DATA_DIR     = "./data",
    USE_SYNTHETIC = False,  # True -> run counting/energy on random tensors, no downloads
    SYN_TRAIN    = 512,     # synthetic dataset sizes (only used if USE_SYNTHETIC)
    SYN_TEST     = 256,
    SYN_EPOCHS   = 1,       # keep synthetic runs fast; accuracy is not the point there

    # ---- compute budget (Colab free tier) -------------------------------------
    QUICK        = False,   # True -> 2 seeds x 10 epochs. Use for a first pass; ~2x faster.
    RUN_VISION   = True,    # False -> skip Section 7 entirely (saves the ~1 GB download)
    NMNIST_TRAIN = 10000,   # subsample N-MNIST. Full is 60000 -> a 3.5 GB tensor that
    NMNIST_TEST  = 2000,    # will exhaust Colab RAM. 10k/2k is plenty for this comparison.
)
if CFG["QUICK"]:
    CFG["SEEDS"], CFG["EPOCHS"] = [0, 1], 10

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
os.makedirs(CFG["DATA_DIR"], exist_ok=True)
print("device:", DEV)
print("USE_SYNTHETIC:", CFG["USE_SYNTHETIC"])
print(json.dumps({k: v for k, v in CFG.items() if k != "SEEDS"}, indent=2))

---
### Setup 2c · What a seed is, and every training parameter *(so nothing is hidden)*

> Two questions a judge will ask: *"what's a seed?"* and *"did you tune the spiking models more
> than the ANN?"* Both answers are here.

#### What a seed is, and why there are three

A **seed** fixes every random choice a run makes: the initial weight values, and the order the
`DataLoader` shuffles the training data. Running the *same architecture* under seeds 0, 1 and 2
gives three independent training runs of an identical network — different starting points,
different data order, everything else the same. Every `accuracy_mean ± accuracy_std` below is
over those three.

`set_seed(s)` covers all four generators that matter: Python's `random`, NumPy, PyTorch on CPU,
and PyTorch on CUDA.

**What three seeds can and cannot tell you.** They catch gross instability — if one seed lands
twenty points below the others, something is wrong. They are **too few for a significance test**.
When two configurations differ by less than about one standard deviation (S4a `0.856 ± 0.020`
vs S4b `0.866 ± 0.004`), read that as *"indistinguishable at this sample size"*, not as a win.

#### Every training parameter — identical for all eleven models

| Parameter | Value | Note |
|---|---|---|
| Optimiser | **Adam** | `torch.optim.Adam` |
| Learning rate | **2e-3** | `CFG["LR"]` |
| Epochs | **20** | 10 if `QUICK` |
| Batch size | **128** | |
| Loss | **CrossEntropyLoss** | logits + integer labels |
| Seeds | **[0, 1, 2]** | 2 if `QUICK` |
| Hidden width | **128** | every model |
| Timesteps `T` | **25** | every model, both dataset pairs |

#### The spiking-neuron parameters

`snn.Leaky` **is** a LIF layer. Verified against **snnTorch 1.0.0** source:

| Parameter | Value | Where from |
|---|---|---|
| `beta` — membrane decay per step | **0.9** | `CFG["BETA"]` — my choice |
| `threshold` | **1.0** | snnTorch default |
| `reset_mechanism` | **`"subtract"`** | snnTorch default — on firing, the threshold is *subtracted* from the membrane rather than the membrane being zeroed |
| `spike_grad` | `fast_sigmoid`, **slope 25** | snnTorch default slope |
| `learn_beta` / `learn_threshold` | **False** | snnTorch defaults — β and threshold are fixed, not learned |

#### What is deliberately absent

No dropout, no batch normalisation, no weight decay, no learning-rate schedule, no early
stopping, no data augmentation, and **no per-architecture tuning of any kind**.

That is both a feature and a limitation, and both should be stated:

- **Feature.** Nothing was tuned to favour one architecture. Every model got an identical budget,
  so no result here comes from me trying harder on my preferred side.
- **Limitation.** The spiking models are therefore probably below their achievable ceiling.
  Published SHD results reach 0.80–0.90 with recurrent SNNs; mine reaches 0.696. Some of that gap
  is architecture (§5b argues this, and the tight 3-seed error bars support it) — but some is
  very likely the absence of tuning.

> **The two biases run in opposite directions, and I do not claim they cancel.** The energy model
> *flatters* the SNN (five of seven omissions favour it, §8.3-equivalent). The training protocol
> *penalises* the SNN (no tuning, and spiking networks are the harder ones to train). Anyone
> quoting a number from this notebook should know both.

---
## Setup cell 2b — save everything to Google Drive (strongly recommended)

**The problem this solves.** Colab disconnects. When it does, two different things happen and they need different fixes:

| What | Lost on disconnect? | Fix |
|---|---|---|
| **Cell outputs** (printed text, plots) | **No** — they live in the notebook file | Save a copy to Drive; the outputs come with it |
| **Variables** (`RESULTS`, trained models) | **Yes** — the Python process is gone | Checkpoint them to Drive (this cell) |
| **Downloaded/decoded datasets** | **Yes** — the local disk is wiped | Put `DATA_DIR` on Drive (this cell) |

So the expensive things — the HD archive, the decoded mel cache, the SHD frame cache, and your training results — get written to your Drive and **survive a disconnect**. On a re-run they are found in the cache and load in seconds instead of being rebuilt.

**How to also keep the outputs themselves.** Either is fine:
- **File → Save a copy in Drive** — saves the whole notebook *including all rendered outputs*, so you can reopen and read the results later without re-running anything.
- **File → Download → Download .ipynb** — same thing, to your own machine.

Set `USE_DRIVE = False` if you would rather keep everything on the temporary Colab disk (fine for a quick synthetic-mode run, bad for a real training run).

In [ ]:
# ---- persist data + results to Google Drive so a disconnect costs you nothing ----
USE_DRIVE = True          # set False to keep everything on the ephemeral Colab disk

PROJECT_DIR = "."
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")                      # will ask you to authorise once
        PROJECT_DIR = "/content/drive/MyDrive/hybrid_ann_snn"
        os.makedirs(PROJECT_DIR, exist_ok=True)
        CFG["DATA_DIR"] = os.path.join(PROJECT_DIR, "data")   # caches now live on Drive
        os.makedirs(CFG["DATA_DIR"], exist_ok=True)
        print("Drive mounted.")
    except Exception as e:
        print("Drive not available (are you on Colab?):", type(e).__name__, e)
        print("Falling back to local disk -- caches will NOT survive a disconnect.")

RESULTS_JSON = os.path.join(PROJECT_DIR, "results_checkpoint.json")
print("project dir :", PROJECT_DIR)
print("data cache  :", CFG["DATA_DIR"])
print("results file:", RESULTS_JSON)

# show what is already cached, so you know what a re-run will skip
import glob as _glob
_cached = sorted(_glob.glob(os.path.join(CFG["DATA_DIR"], "*.pt")))
if _cached:
    print("\nalready cached (these will load instantly, not rebuild):")
    for f in _cached:
        print(f"   {os.path.basename(f):32s} {os.path.getsize(f)/1e6:8.1f} MB")
else:
    print("\nno caches yet -- the first real-data run will build and save them here.")

---
## Setup cell 3 — the two energy models

Both models from the front matter, in code. The self-test at the bottom reproduces Dampfhoffer's **published** figures (break-even 1.38, arithmetic ~0.8% of SNN energy, memory ~96% of ANN energy). **If those assertions ever fail, stop — the accounting is wrong and every downstream number is meaningless.**

- **Model (a), operation count:** two constant sets (FP32 and INT32), which disagree ~6× in ratio, so a conclusion that depends on the choice is exposed.
- **Model (b), memory-inclusive:** the Dampfhoffer per-synapse costs derived by hand in §2.2, normalised to one MAC.

In [ ]:
# ---------- Model (a): operation count only ----------
OPCOUNT = {
    "fp32_45nm":  {"E_MAC": 4.6e-12, "E_AC": 0.9e-12},   # Chen et al. arXiv:2301.11929 [READ]
    "int32_45nm": {"E_MAC": 3.2e-12, "E_AC": 0.1e-12},   # Datta et al. arXiv:2107.12374 Tbl IV [READ]
}
def energy_opcount(macs, acs, variant="fp32_45nm"):
    c = OPCOUNT[variant]
    return macs * c["E_MAC"] + acs * c["E_AC"]

# ---------- Model (b): memory-inclusive (Dampfhoffer, normalised to 1 MAC) ----------
E_AC_N, E_MAC_N, E_RW_N = 0.13, 1.0, 5.4
ANN_SYN  = 4 * E_RW_N + E_MAC_N     # 22.6  = read iact+weight+psum, write psum, MAC
SNN_SYN  = 3 * E_RW_N + E_AC_N      # 16.33 = read weight+state, write state, AC (no iact read)
LIF_LEAK = 2 * E_RW_N + E_MAC_N     # 11.8  = read state, write state, decay MAC (per neuron per step)

def energy_memory(mac_syn_ops, ac_syn_ops, lif_neuron_steps=0):
    """Normalised units (1 = one MAC). lif_neuron_steps = (# LIF neurons)*T.
    A part of the network with no spiking neurons (e.g. a pure ANN or an ANN
    readout) contributes 0 here, because it has no membrane to update."""
    return mac_syn_ops * ANN_SYN + ac_syn_ops * SNN_SYN + lif_neuron_steps * LIF_LEAK

# ---------- self-test against the paper's published numbers ----------
be = ANN_SYN / SNN_SYN
assert abs(be - 1.38) < 0.01,                 f"break-even {be:.3f} != 1.38"
assert abs(E_AC_N / SNN_SYN - 0.008) < 0.002, "arithmetic share should be ~0.8%"
assert abs((4 * E_RW_N) / ANN_SYN - 0.956) < 0.01, "ANN memory share should be ~96%"
print(f"ANN synapse   {ANN_SYN:6.2f}   (read iact+weight+psum, write psum, +MAC)")
print(f"SNN synapse   {SNN_SYN:6.2f}   (read weight+state, write state, +AC; no iact read)")
print(f"LIF leak      {LIF_LEAK:6.2f}   per neuron per timestep")
print(f"break-even    {be:6.3f}   spikes/synapse   [paper: 1.38]  PASS")
print(f"arithmetic is {E_AC_N/SNN_SYN*100:5.2f}% of SNN energy   [paper: ~1%]  PASS")
print(f"memory is     {(4*E_RW_N)/ANN_SYN*100:5.1f}% of ANN energy   [paper: 96%] PASS")

---
## Setup cell 3b — a *second* ANN accelerator: ideal data reuse

**Why this exists.** The `22.6` above is the paper's **naive worst case** — an accelerator that
refetches everything for every single operation. Real accelerators read a weight once and reuse
it locally many times; an event-driven SNN cannot, because spikes arrive in an unpredictable
order. Costing my well-designed SNN *only* against a badly-designed ANN is exactly the
design-effort gap flagged in §2.0. This cell closes it.

**Why the constants change.** Table I has **no register-file cost at 45 nm** — that entry is
literally a slash (`/`). Equation 11 needs one. That is why the paper itself switches to
65 nm / 16-bit for its reuse analysis, and why this cell does too. Everything in it is
internally consistent at 65 nm / 16-bit, and the break-even barely moves
(`25 ÷ 18.06 = 1.384` against the 45 nm `1.384`) — a useful stability check on the whole method.

**The reuse factors** for a fully-connected layer come straight from the paper:
`RF_iact = n_out`, `RF_psum = n_in`, and **`RF_weight = 1` — no weight reuse at all.**
That last one turns out to be the whole story; see the note under the output.

In [ ]:
# ---------- Model (c): ANN with IDEAL DATA REUSE  (Dampfhoffer Eq. 11, 65nm / 16-bit) ----------
# Table I, 65nm/16bit row:   E_AC 0.06 | E_MAC 1 | E_R/W 6 | E_R/W_reg 1
# (The 45nm/8bit row has NO register-file entry -- that is why this model changes node.)
E_AC_6, E_MAC_6, E_RW_6, E_REG_6 = 0.06, 1.0, 6.0, 1.0

ANN_SYN_NAIVE_6 = 4 * E_RW_6 + E_MAC_6      # 25.00  -- same structure as the 45nm 22.6
SNN_SYN_6       = 3 * E_RW_6 + E_AC_6       # 18.06
LIF_LEAK_6      = 2 * E_RW_6 + E_MAC_6      # 13.00

def fc_reuse_cost(n_in, n_out):
    """Eq. 11 per-synapse cost for a FULLY-CONNECTED layer with ideal data reuse.
    Paper's FC reuse factors: RF_iact = n_out, RF_psum = n_in, RF_weight = 1.
    RF_weight = 1 is not a simplification: in an FC layer each weight is used exactly
    once per forward pass, so it must be fetched every time -- just like the SNN."""
    return (E_RW_6 / n_out           # iact  read, reused across n_out neurons
          + E_RW_6 / 1               # weight read, NO reuse in an FC layer
          + 2 * E_RW_6 / n_in        # psum  read + write, reused across n_in inputs
          + 4 * E_REG_6              # local register-file traffic (iact, psum r/w, weight)
          + E_MAC_6)                 # the multiply-accumulate itself

def topo_reuse_cost(layers):
    """Synapse-weighted average Eq. 11 cost over a list of (n_in, n_out) FC layers."""
    tot = sum(a * b for a, b in layers)
    return sum(a * b * fc_reuse_cost(a, b) for a, b in layers) / tot

def energy_reuse(mac_syn_ops, ac_syn_ops, lif_neuron_steps, layers):
    """Same measured op-counts, but MACs priced on an IDEAL-REUSE ANN accelerator."""
    return (mac_syn_ops * topo_reuse_cost(layers)
          + ac_syn_ops  * SNN_SYN_6
          + lif_neuron_steps * LIF_LEAK_6)

# ---------- self-test: reproduce the paper's OWN published ideal-reuse target ----------
# At convolutional-scale reuse factors (RF ~ 1e2-1e3, paper Fig. 4) the distant-memory terms
# nearly vanish and Eq. 11 collapses to the register-file floor. The paper publishes a target
# sparsity of 0.28 for that case (Table III, "Ideal reuse"). If this assertion fails, the
# implementation is wrong and every number below it is meaningless.
# NOTE: RF_weight is ~500 here, NOT 1 -- a CONVOLUTIONAL layer reuses each weight
# hundreds of times. That single term is the whole difference between conv and FC.
_conv = E_RW_6/500 + E_RW_6/500 + 2*E_RW_6/500 + 4*E_REG_6 + E_MAC_6
assert abs(ANN_SYN_NAIVE_6 / SNN_SYN_6 - 1.38) < 0.01, "naive break-even should still be 1.38"
assert abs(_conv / SNN_SYN_6 - 0.28)      < 0.01, f"ideal-reuse target {_conv/SNN_SYN_6:.3f} != 0.28"

print(f"naive ANN synapse (65nm/16b)   {ANN_SYN_NAIVE_6:6.2f}")
print(f"SNN synapse       (65nm/16b)   {SNN_SYN_6:6.2f}")
print(f"LIF leak          (65nm/16b)   {LIF_LEAK_6:6.2f}")
print(f"break-even, naive ANN          {ANN_SYN_NAIVE_6/SNN_SYN_6:6.3f}   [paper: 1.38]  PASS")
print(f"break-even, conv-scale reuse   {_conv/SNN_SYN_6:6.3f}   [paper: 0.28]  PASS\n")

print(f"{'topology':<26s}{'ANN/syn':>9s}{'x cheaper':>11s}{'break-even':>12s}{'  at T=25':>10s}")
for _nm, _ly in [("HD / SHD@40  40-128-20",  [(40,128),(128,20)]),
                 ("SHD         700-128-20",  [(700,128),(128,20)]),
                 ("MNIST       784-128-10",  [(784,128),(128,10)]),
                 ("N-MNIST    2312-128-10",  [(2312,128),(128,10)])]:
    _c = topo_reuse_cost(_ly)
    print(f"{_nm:<26s}{_c:9.2f}{ANN_SYN_NAIVE_6/_c:11.2f}{_c/SNN_SYN_6:12.3f}{25*_c/SNN_SYN_6:10.2f}")

**The result to take from this cell — and it cuts against my own topology choice.**

An ideal-reuse ANN is only **2.2–2.3× cheaper** than the naive one on *these* networks — not the
3.3–3.7× the paper's Eyeriss figures imply. The entire difference between my `11.07` and the
paper's convolutional `5.05` is the **weight read**.

> In a fully-connected layer each weight is used exactly once per forward pass, so it must be
> fetched every time — **exactly like the SNN**. Convolution reuses each weight hundreds of times
> and hands that discount to the ANN *alone*, because an event-driven SNN cannot predict the order
> its spikes will arrive in.

So my MLPs sit at the end of the reuse axis **most favourable to spiking**. Every margin reported
under the ideal-reuse model would shrink — possibly reverse — on a convolutional network. That is
a limitation of the topology, and it should be quoted wherever these numbers are.

Note also that the break-even is ~15.3–15.7 at `T = 25`, against `34.5` for the naive ANN. All
three audio spike rates (5.46, 6.02, 11.23) still clear it; the 40-channel control clears it by
only 1.4×.

---
## Setup cell 4 — operation counting

The counting rule that decides everything (NeuroBench's convention):

> A synapse whose **driving activation is binary** costs an **AC**. A synapse whose driving activation is **real-valued** costs a **MAC**.

Consequences, all foreshadowed in §2:

- On **SHD** the input is already spikes → layer 1 is **ACs** (the event-native advantage).
- On **HD** the input is a real-valued spectrogram → layer 1 is **MACs**, unavoidably (§2.7).
- An **ANN readout after an accumulator runs once**, not `T` times → why SNN-front hybrids (§3) are cheap.
- An **ANN front end on a time series runs `T` times** → why ANN-front hybrids (§4) are expensive.

`N_spikes/syn` for a layer = (total spikes from the previous layer, summed over all timesteps) ÷ (number of presynaptic neurons). All counts come from **real execution traces** in the evaluation pass, never from architecture arithmetic.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class OpStats:
    mac_syn: float = 0.0          # MAC-type synaptic operations
    ac_syn:  float = 0.0          # AC-type synaptic operations
    lif_steps: float = 0.0        # N_neur * T   (LIF membrane updates only)
    spikes: dict = field(default_factory=dict)   # layer -> summed spikes/neuron
    n_syn: int = 0
    n_neur: int = 0

    def per_sample(self, n):
        return OpStats(self.mac_syn/n, self.ac_syn/n, self.lif_steps/n,
                       {k: v/n for k, v in self.spikes.items()},
                       self.n_syn, self.n_neur)

    def summary(self, label=""):
        e_op_fp32  = energy_opcount(self.mac_syn, self.ac_syn, "fp32_45nm")
        e_op_int32 = energy_opcount(self.mac_syn, self.ac_syn, "int32_45nm")
        e_mem      = energy_memory(self.mac_syn, self.ac_syn, self.lif_steps)
        return dict(model=label,
                    MACs=round(self.mac_syn), ACs=round(self.ac_syn),
                    lif_steps=round(self.lif_steps),
                    N_syn=self.n_syn, N_neur=self.n_neur,
                    syn_per_neur=round(self.n_syn/max(self.n_neur, 1), 1),
                    E_opcount_fp32_nJ=round(e_op_fp32*1e9, 4),
                    E_opcount_int32_nJ=round(e_op_int32*1e9, 4),
                    E_memory_norm=round(e_mem, 1),
                    # spikes per SYNAPSE per inference = spikes emitted by the presynaptic
                    # layer, divided by that layer's width. Compare against the break-even
                    # in section 2.3 -- but read the "T passes" caveat there first.
                    **{f"spk/syn_{k}": round(v, 4) for k, v in self.spikes.items()})

def is_binary(t):
    """True if the tensor holds only 0/1 -> its synapses are ACs."""
    u = torch.unique(t)
    return bool(((u == 0) | (u == 1)).all())

---
# Understanding the four datasets — and how to explain them to an audience

*Read this before the loader cells. If you have to present this project, this section is the part you should be able to say out loud without notes. Every dataset below has a clickable link to its official page, most with figures and example recordings.*

The project needs, for each domain, **two views of the same underlying data**: one that arrives as **real-valued numbers** (which an ANN eats naturally) and one that arrives as **spikes** (which an SNN eats naturally). That pairing is what makes an ANN-vs-SNN comparison fair — otherwise you are comparing two different tasks and calling it a hardware result.

| Domain | "Numbers" version (→ ANN) | "Spikes" version (→ SNN) | Classes |
|---|---|---|---|
| **Audio** | **HD** — Heidelberg Digits (raw audio) | **SHD** — Spiking Heidelberg Digits | 20 |
| **Vision** | **MNIST** — handwritten digit images | **N-MNIST** — Neuromorphic MNIST | 10 |

---

## A · HD and SHD (audio) — and what "700 channels" actually means

**HD (Heidelberg Digits)** is about 10,000 studio recordings of people saying the digits **0–9 in both English and German**, which is why there are **20 classes**, not 10. Twelve speakers; two of them appear only in the test set, so the test set genuinely measures generalisation to unheard voices.

**SHD (Spiking Heidelberg Digits)** is *the very same recordings*, passed through an artificial cochlea model called **Lauscher**, which converts the audio into spike trains. This is the crucial point for the project: **SHD is not a different task from HD — it is a different encoding of the identical audio.** That is exactly the controlled pair we need.

### How to explain "700 channels" to an audience

This is the question you will most likely be asked, so here is an explanation that works:

> *"Your inner ear contains a spiral organ, the cochlea. Along its length, different positions respond to different pitches — one end to high frequencies, the other to low ones, like a piano keyboard rolled into a spiral. Each position has hair cells that fire nerve impulses when their frequency is present in the sound.*
>
> *The Lauscher model simulates 700 of those positions. So a **channel** is one simulated listening-post tuned to one narrow frequency band. When the sound contains that frequency, that channel fires a spike. The dataset is therefore 700 rows of spike trains: row 1 is the lowest-frequency listener, row 700 the highest, and reading down the rows at one instant tells you what frequencies were present at that moment."*

Two consequences worth stating explicitly, because they drive results in this notebook:

- **A "channel" is an input neuron.** 700 channels means the first layer has 700 inputs, so with 128 hidden units the first weight matrix has 700 × 128 = 89,600 synapses. That is a *big* first layer, and Section 2's energy result turns on exactly this.
- **Downsampling channels is not free.** Many papers reduce 700 → 140 or 40 channels for speed. That throws away frequency resolution (accuracy drops) *and* changes the synapse-to-neuron ratio that governs whether the leaky neuron is affordable (§2.5). Section 5 measures that cost deliberately instead of hiding it.

**Links:**
- [Zenke Lab — official SHD/SSC page](https://zenkelab.org/resources/spiking-heidelberg-datasets-shd/) — the canonical source, with dataset description and download code **[READ]**
- [IEEE DataPort — Heidelberg Spiking Datasets](https://ieee-dataport.org/open-access/heidelberg-spiking-datasets) — dataset record and licence **[READ]**
- [snnTorch docs — SHD loader](https://snntorch.readthedocs.io/en/latest/snntorch.spikevision.spikedata.html) — confirms shape `[num_steps × 700]`, 8156 train / 2264 test **[READ]**
- [Papers-with-Code SHD leaderboard](https://paperswithcode.com/sota/audio-classification-on-shd) — what accuracies are actually achievable, useful for judging whether your model is competitive
- Original paper: Cramer, Stradmann, Schemmel & Zenke, *The Heidelberg Spiking Data Sets*, IEEE TNNLS 2020

> **A discrepancy worth knowing about (and mentioning if asked).** The official page lists the split as **8,332 train / 2,088 test**, while `tonic` and snnTorch report **8,156 / 2,264**. Both total 10,420 — the difference is where the boundary is drawn. Report whatever your loader actually returns, and say so.

---

## B · MNIST and N-MNIST (vision) — and what "34×34, two polarities" means

**MNIST** needs little introduction: 70,000 handwritten digits, 28×28 greyscale pixels, 10 classes. Each pixel is a **real number** (brightness), so an ANN's first layer must multiply — this is the "arrives as numbers" case.

**N-MNIST (Neuromorphic MNIST)** was made by Orchard et al. (2015). They displayed each MNIST digit on a screen and pointed an **event camera** (an ATIS sensor) at it — then **moved the camera** in three short, pre-programmed sweeps called **saccades**, named after the flicks your own eyes make several times a second.

### How to explain an event camera to an audience

> *"A normal camera takes complete pictures at a fixed rate — say 30 frames a second — whether or not anything changed. An event camera doesn't take pictures at all. Each pixel watches its own patch of the world independently and stays silent until the brightness there changes. When it does, that pixel immediately sends a tiny message: 'I am pixel (12, 20), I got brighter, at this microsecond.' Nothing moving means almost no data. It is the same trick as a spiking neuron: stay quiet unless there is news."*

So the pieces of "34×34, two polarities" are:

- **34×34** — the pixel grid. It is bigger than MNIST's 28×28 because the camera *moves*, so the digit sweeps across a slightly larger area than it occupies at rest.
- **Two polarities** — each pixel reports two kinds of event: **ON** (got brighter) and **OFF** (got dimmer). These are kept as separate channels, because "brightening" and "dimming" mean different things.
- Flattened for a fully-connected network: 34 × 34 × 2 = **2,312 input channels** — considerably wider than MNIST's 784, which matters for the energy accounting.
- Each recording lasts ~**300 ms** (three saccades of ~105 ms), and holds a few thousand events.

**Links:**
- [Garrick Orchard — official N-MNIST page](https://www.garrickorchard.com/datasets/n-mnist) — downloads, format description, and **videos showing the saccadic motion** (the best thing to show an audience) **[READ]**
- [Orchard et al. 2015, *Converting Static Image Datasets to Spiking Neuromorphic Datasets Using Saccades*, Frontiers in Neuroscience](https://doi.org/10.3389/fnins.2015.00437) — the original paper **[READ, via summaries]**
- [Neuromorphic Systems — N-MNIST format notes](https://neuromorphicsystems.github.io/land/n-mnist) — the binary event format, if you need to parse it yourself **[READ]**
- [Yann LeCun — the original MNIST page](http://yann.lecun.com/exdb/mnist/)

> ### The caveat that makes this project interesting — say this one out loud
>
> There is a **published, peer-reviewed challenge** to whether N-MNIST is really "neuromorphic" in a meaningful sense: [*Is Neuromorphic MNIST Neuromorphic? Analyzing the Discriminative Power of Neuromorphic Datasets in the Time Domain*](https://www.frontiersin.org/journals/neuroscience/articles/10.3389/fnins.2021.608567/full) (Frontiers in Neuroscience, 2021; [arXiv preprint](https://arxiv.org/abs/1807.01013)) **[READ]**. The authors point out that the saccades are a camera sweeping a **static picture** in a pre-programmed way — so it is fair to ask how much *genuinely discriminative* information the time axis carries at all.
>
> **This is not a footnote for us — it is central.** The project's deepest hypothesis is that spiking pays off in proportion to how genuinely temporal the signal is. SHD is truly temporal (speech unfolds in time and cannot be reordered). N-MNIST's time axis is camera motion over a still digit. So the audio and vision pairs are not just two datasets — they are a **designed contrast**, and Section 7 uses that contrast deliberately.

---

## C · Why four datasets rather than two

Using only one pair could not separate two very different kinds of claim:

- **Structural claims** — e.g. *"running the dense layer once instead of T times is cheaper"* — follow from operation counting and should hold on **any** dataset. If they hold on both pairs, they are claims about hybrid architectures.
- **Signal-dependent claims** — e.g. *"the event-native SNN is efficient"* — depend on sparsity and on whether time carries information. These may hold on audio and fail on static vision.

A single dataset cannot tell those apart. Two contrasting pairs can. That is the whole reason Section 7 exists.

---
## Setup cell 5 — data loaders (synthetic fallback + SHD + HD)

Three ways to get data, chosen by `CFG["USE_SYNTHETIC"]`:

- **Synthetic** (`True`): random tensors of the right shape and binarisation. SHD-style inputs are sparse 0/1; HD-style inputs are real-valued. Accuracy will sit near chance — the point is that **operation counts and energy are exact and real** even with no downloads.
- **SHD** (`False`): loaded via `tonic`. Two things matter and are handled below: bin the events into `T` frames, then **binarise** with `(frames > 0)`. `ToFrame` returns event *counts*, not spikes — without binarising, layer 1 would be miscounted as MACs and the entire event-native advantage would silently vanish. This is the single easiest way to get wrong numbers.
- **HD** (`False`): the raw-audio parent of SHD, fetched and turned into log-mel spectrograms in the next cells.

> Reported SHD split: 8,156 train / 2,264 test, 20 classes, 700 channels. One published source lists 8,332 / 2,088 — both total 10,420. Whatever your loader returns is what gets reported.
>
> **Known tonic bug, fixed here automatically.** SHD stores spike times as `float16` (max value 65504), but tonic converts seconds→microseconds with `times * 1e6`, which overflows float16 to `inf` for every sample (0.7 × 1e6 = 700000 ≫ 65504). Symptom: `RuntimeWarning: overflow encountered in cast` / `invalid value encountered in multiply`, and silently corrupt spike times. The loader below monkey-patches tonic to upcast the times to `float64` before the multiply, which fixes it cleanly. You do not need to do anything — the patch is applied automatically when SHD loads. (Your downloaded file is fine; this is a dtype bug in tonic, not a bad download.)

In [ ]:
from torch.utils.data import DataLoader, Dataset, TensorDataset
import time, sys

# ---------- synthetic ----------
class SyntheticDS(Dataset):
    """Random data of the correct shape. spike_input=True -> sparse 0/1; else real-valued."""
    def __init__(self, n, T, n_ch, n_classes, spike_input, rate=0.05, seed=0):
        g = torch.Generator().manual_seed(seed)
        if spike_input:
            self.X = (torch.rand(n, T, n_ch, generator=g) < rate).float()
        else:
            self.X = torch.randn(n, T, n_ch, generator=g)
        self.Y = torch.randint(0, n_classes, (n,), generator=g)
    def __len__(self):  return len(self.Y)
    def __getitem__(self, i): return self.X[i], int(self.Y[i])

def synth_loaders(n_ch, spike_input, T=None, batch=None):
    T = T or CFG["T"]; batch = batch or CFG["BATCH"]
    tr = SyntheticDS(CFG["SYN_TRAIN"], T, n_ch, CFG["N_CLASSES"], spike_input, seed=0)
    te = SyntheticDS(CFG["SYN_TEST"],  T, n_ch, CFG["N_CLASSES"], spike_input, seed=1)
    L = lambda d, s: DataLoader(d, batch_size=batch, shuffle=s, drop_last=False)
    return L(tr, True), L(te, False)

# ---------- SHD (real) ----------
# WHY THIS WAS REWRITTEN (v5 -> v6)
# The previous version wrapped tonic's Dataset and let the DataLoader call
# __getitem__ once per sample, per epoch. Each of those calls re-opened the
# ~80 MB HDF5 file and ran tonic's per-sample ToFrame in Python: measured at
# 4.2 ms/sample, i.e. ~35 s for one pass over the 8156 training samples.
# Section 2 alone is 3 seeds x 20 epochs = 60 passes -> ~35 min of pure data
# loading, with no output printed until the very end. It was never stuck; it
# was silent and quadratically wasteful, decoding the same data 60 times.
#
# THE FIX: decode the whole dataset ONCE into a uint8 tensor [N, T, C]
# (143 MB train / 40 MB test at 700 channels -- fits in Colab RAM easily),
# cache it to disk like the HD loader already does, then train from memory.
# The frame-building below is a vectorised reimplementation of
# tonic's Downsample + ToFrame(n_time_bins=T); it was verified to produce
# BIT-IDENTICAL binarised frames to the old path at both 700 and 40 channels.

def _shd_h5_paths(root):
    """Instantiate tonic's SHD once (no transform) purely to trigger the
    download/extract, then hand back the raw .h5 paths."""
    import tonic
    tr = tonic.datasets.SHD(save_to=root, train=True)
    te = tonic.datasets.SHD(save_to=root, train=False)
    p = lambda d: os.path.join(d.location_on_system, d.data_filename)
    return p(tr), p(te)

def _build_shd_frames(h5_path, T, n_channels, sensor_ch=700, desc=""):
    """Vectorised equivalent of tonic Downsample->ToFrame->binarise.
    Returns X uint8 [N, T, C] and Y int64 [N]."""
    import h5py
    from tqdm.auto import tqdm
    f = h5py.File(h5_path, "r")
    times_ds, units_ds = f["spikes/times"], f["spikes/units"]
    labels = np.asarray(f["labels"]).astype(np.int64)
    N = len(labels)
    out = np.zeros((N, T, n_channels), dtype=np.uint8)
    sf = n_channels / sensor_ch
    for i in tqdm(range(N), desc=desc, unit="sample", file=sys.stdout):
        # seconds (float16 in the file) -> float64 BEFORE *1e6: this is the
        # same overflow fix as before, float16 max is 65504 and 0.7*1e6 is inf.
        t = (np.asarray(times_ds[i]).astype(np.float64) * 1e6).astype(np.int64)
        u = np.asarray(units_ds[i]).astype(np.int64)
        if sf != 1.0:
            u = (u * sf).astype(np.int64)      # matches tonic's int truncation
        if t.size == 0:
            continue
        width = (t[-1] - t[0]) // T            # tonic floors the bin width
        idx = np.full(t.size, -1, dtype=np.int64)
        if width > 0:
            ws = np.arange(T) * width + t[0]
            we = ws + width
            s = np.searchsorted(t, ws); e = np.searchsorted(t, we)
            for k in range(T):
                idx[s[k]:e[k]] = k             # tail events past T*width dropped, as tonic does
        else:
            idx[:] = 0
        v = idx >= 0
        flat = idx[v] * n_channels + np.clip(u[v], 0, n_channels - 1)
        cnt = np.bincount(flat, minlength=T * n_channels).reshape(T, n_channels)
        out[i] = (cnt > 0).astype(np.uint8)    # <-- BINARISE (events -> spikes)
    f.close()
    return torch.from_numpy(out), torch.from_numpy(labels)

class SHDMem(Dataset):
    """In-memory frames. Stored uint8 to save RAM, cast to float per item."""
    def __init__(self, X, Y): self.X, self.Y = X, Y
    def __len__(self):        return len(self.Y)
    def __getitem__(self, i): return self.X[i].float(), int(self.Y[i])

def shd_cached(n_channels, T):
    """Decode once, cache to ./data/shd_{C}_{T}.pt, reuse thereafter."""
    cache = os.path.join(CFG["DATA_DIR"], f"shd_{n_channels}_{T}.pt")
    if os.path.exists(cache):
        d = torch.load(cache)
        print(f"  [SHD cache hit] {cache}  train {tuple(d['Xtr'].shape)}  test {tuple(d['Xte'].shape)}",
              flush=True)
        return d["Xtr"], d["Ytr"], d["Xte"], d["Yte"]
    print(f"  [SHD cache miss] building {n_channels}ch x {T} frames -- one time only", flush=True)
    t0 = time.time()
    p_tr, p_te = _shd_h5_paths(CFG["DATA_DIR"])
    Xtr, Ytr = _build_shd_frames(p_tr, T, n_channels, desc="SHD train -> frames")
    Xte, Yte = _build_shd_frames(p_te, T, n_channels, desc="SHD test  -> frames")
    torch.save({"Xtr": Xtr, "Ytr": Ytr, "Xte": Xte, "Yte": Yte}, cache)
    mb = (Xtr.numel() + Xte.numel()) / 1e6
    print(f"  [SHD cached] {cache}  {mb:.0f} MB  in {time.time()-t0:.1f}s", flush=True)
    return Xtr, Ytr, Xte, Yte

def shd_loaders(n_channels, T=None, batch=None):
    T = T or CFG["T"]; batch = batch or CFG["BATCH"]
    if CFG["USE_SYNTHETIC"]:
        return synth_loaders(n_channels, spike_input=True, T=T, batch=batch)
    Xtr, Ytr, Xte, Yte = shd_cached(n_channels, T)
    # num_workers=0 on purpose: data is already in RAM, so worker processes
    # would only add IPC overhead (and h5py + fork is a known hang source).
    L = lambda X, Y, s: DataLoader(SHDMem(X, Y), batch_size=batch, shuffle=s,
                                   num_workers=0, drop_last=False)
    return L(Xtr, Ytr, True), L(Xte, Yte, False)

# quick check
if CFG["USE_SYNTHETIC"]:
    _tr, _te = shd_loaders(CFG["SHD_CHANNELS"])
    _x, _y = next(iter(_tr))
    print("SYNTHETIC SHD-style batch:", tuple(_x.shape), "binary?", is_binary(_x),
          "| mean spikes/neuron/step:", round(float(_x.mean()), 4))
else:
    print("USE_SYNTHETIC is False -> SHD decodes once into a cached tensor on first use.")


### HD loader, step 1 — inspect before parsing

HD is SHD's raw-audio parent: the same ~10,420 recordings before the cochlea model, 12 speakers, digits 0–9 in English and German → 20 classes. Archive: `https://compneuro.net/datasets/hd_audio.tar.gz`.

This cell only downloads and prints the archive's structure; nothing is parsed yet. The internal layout is not known in advance, so **read the printout, then confirm the next cell's parser matches what you actually see.** Skipped entirely under `USE_SYNTHETIC`.

> **Not verified this session:** the archive's internal layout and the parser below were not run against the real download (the host was not reachable from this environment). They are carried over parse-clean but *unexecuted on real data* — eyeball the Cell-6 output before trusting Cell-7.

In [ ]:
import glob, re, collections

def download_and_inspect_hd():
    import urllib.request, tarfile
    HD_URL = "https://compneuro.net/datasets/hd_audio.tar.gz"
    HD_TGZ = os.path.join(CFG["DATA_DIR"], "hd_audio.tar.gz")
    HD_DIR = os.path.join(CFG["DATA_DIR"], "hd_audio")
    if not os.path.exists(HD_TGZ):
        print("downloading HD audio (~few hundred MB) -- progress below", flush=True)
        def _hook(blocks, bs, total):
            if total > 0 and (blocks % 200 == 0 or blocks * bs >= total):
                pct = min(100.0, 100.0 * blocks * bs / total)
                print(f"\r   {pct:5.1f}%  ({blocks*bs/1e6:6.1f} / {total/1e6:.1f} MB)",
                      end="", flush=True)
        urllib.request.urlretrieve(HD_URL, HD_TGZ, reporthook=_hook)
        print(flush=True)
    print("archive:", HD_TGZ, f"{os.path.getsize(HD_TGZ)/1e6:.1f} MB")
    if not os.path.isdir(HD_DIR):
        os.makedirs(HD_DIR, exist_ok=True)
        print("extracting archive (this can take a minute)...", flush=True)
        with tarfile.open(HD_TGZ) as t:
            t.extractall(HD_DIR)
    print("extracted to:", HD_DIR)
    all_files = glob.glob(os.path.join(HD_DIR, "**", "*"), recursive=True)
    files = [f for f in all_files if os.path.isfile(f)]
    exts = collections.Counter(os.path.splitext(f)[1].lower() for f in files)
    print(f"\n{len(files)} files")
    print("extensions:", dict(exts))
    print("\nfirst 12 filenames:")
    for f in sorted(files)[:12]:
        print("   ", os.path.relpath(f, HD_DIR))
    return HD_DIR, files

if CFG["USE_SYNTHETIC"]:
    print("USE_SYNTHETIC is True -> skipping HD download/inspect.")
    HD_DIR, files = None, []
else:
    HD_DIR, files = download_and_inspect_hd()

### HD loader, step 2 — parse labels

The parser tries several strategies in order and reports which one succeeded. Labels must come out as 0–19 (10 digits × 2 languages), matching SHD's class definition.

The **first** pattern (`lang-english..._digit-5`) matches the real HD archive's filename layout, confirmed by running the inspect cell above on the actual download. The remaining patterns are generic fallbacks. If the strategy report ever shows most files unlabelled, inspect the filenames printed in the step-1 cell and edit `LABEL_PATTERNS` — never proceed on a parser you have not eyeballed. Skipped under `USE_SYNTHETIC`.

In [ ]:
import sys
DIGIT_WORDS = {
    "en": ["zero","one","two","three","four","five","six","seven","eight","nine"],
    "de": ["null","eins","zwei","drei","vier","fuenf","fünf","sechs","sieben","acht","neun"],
}
DE_MAP = {"null":0,"eins":1,"zwei":2,"drei":3,"vier":4,"fuenf":5,"fünf":5,
          "sechs":6,"sieben":7,"acht":8,"neun":9}
EN_MAP = {w:i for i,w in enumerate(DIGIT_WORDS["en"])}
LABEL_PATTERNS = [
    # Matches the real HD archive layout (confirmed on Colab): filenames like
    # "..._lang-english_..._digit-5_...". This specific pattern is tried FIRST.
    ("filename lang-/digit-", r"lang-(?P<lang>english|german).*_digit-(?P<digit>\d)"),
    # Generic fallbacks, in case a future archive uses a different convention:
    ("digit+lang tokens", r"(?P<lang>english|german|en|de)[^a-z0-9]*(?P<digit>\d)"),
    ("lang+digit tokens", r"(?P<digit>\d)[^a-z0-9]*(?P<lang>english|german|en|de)"),
    ("digit dir",         r"[/_-](?P<digit>\d)[/_-]"),
]

def label_from_path(p, root):
    s = os.path.relpath(p, root).lower()
    for w, d in {**EN_MAP, **DE_MAP}.items():
        if re.search(rf"(?<![a-z]){re.escape(w)}(?![a-z])", s):
            lang = 1 if w in DE_MAP and w not in EN_MAP else 0
            return d + 10*lang, f"word:{w}"
    for name, pat in LABEL_PATTERNS:
        m = re.search(pat, s)
        if m:
            d = int(m.group("digit")); lang = 0
            if "lang" in m.groupdict() and m.group("lang"):
                lang = 1 if m.group("lang")[:1] in ("g","d") else 0
            return d + 10*lang, name
    return None, None

def parse_hd_labels(files, root):
    from tqdm.auto import tqdm
    audio = sorted([f for f in files if f.lower().endswith((".flac", ".wav"))])
    print(f"{len(audio)} audio files found", flush=True)
    labels, strategies, kept = [], collections.Counter(), []
    for p in tqdm(audio, desc="parsing labels", unit="file", file=sys.stdout):
        lab, how = label_from_path(p, root)
        if lab is not None and 0 <= lab < 20:
            labels.append(lab); kept.append(p); strategies[how] += 1
    print("strategy usage:", dict(strategies))
    print(f"labelled {len(kept)} / {len(audio)} files")
    print("class histogram:", dict(sorted(collections.Counter(labels).items())))
    if len(kept) < 0.8 * len(audio):
        print("\n*** PARSER LIKELY WRONG *** inspect filenames above and edit LABEL_PATTERNS")
    else:
        print("\nparser looks sane -> proceed")
    return kept, labels

if CFG["USE_SYNTHETIC"]:
    print("USE_SYNTHETIC is True -> skipping HD label parse.")
    kept, labels = [], []
else:
    kept, labels = parse_hd_labels(files, HD_DIR)

### HD → mel-spectrogram tensors

Each recording becomes a `[T, N_MELS]` tensor: log-mel features resampled to exactly `T` frames so HD and SHD share a time axis. Cached to disk (decoding ~10k FLAC files takes a few minutes the first time only). Skipped under `USE_SYNTHETIC`.

**The asymmetry to state in any writeup:** HD gets an FFT front end; SHD gets a cochlea model. Both cost real energy and **neither is counted** by any model here (all begin at the first synapse). The "SNN is more efficient on audio" claim quietly assumes cochlear transduction is free.

In [ ]:
import sys
from torch.utils.data import Dataset as _DS

class TensorDS(_DS):
    def __init__(self, X, Y): self.X, self.Y = X, Y
    def __len__(self): return len(self.Y)
    def __getitem__(self, i): return self.X[i], int(self.Y[i])

def build_hd_cache(paths, labs, n_mels, T, sr_target=16000):
    """Decode every HD audio file to a [T, n_mels] log-mel tensor.
    PROGRESS: this decodes ~10k FLAC files and is the slowest cell in the notebook
    (typically 3-8 minutes on Colab). The tqdm bar below shows live progress, the
    running failure count, and an ETA, so a slow run is distinguishable from a hung one."""
    import torchaudio, torchaudio.transforms as AT
    from tqdm.auto import tqdm
    import time as _time
    _t0 = _time.time()
    mel = AT.MelSpectrogram(sample_rate=sr_target, n_fft=400, hop_length=160, n_mels=n_mels)
    db  = AT.AmplitudeToDB()
    X = torch.zeros(len(paths), T, n_mels, dtype=torch.float32); ok = []
    n_fail = 0
    bar = tqdm(list(enumerate(paths)), desc="HD audio -> log-mel", unit="file", file=sys.stdout)
    for i, p in bar:
        try:
            wav, sr = torchaudio.load(p)
            if wav.shape[0] > 1: wav = wav.mean(0, keepdim=True)
            if sr != sr_target: wav = AT.Resample(sr, sr_target)(wav)
            m = db(mel(wav)).squeeze(0)
            m = torch.nn.functional.interpolate(m.unsqueeze(0), size=T,
                    mode="linear", align_corners=False).squeeze(0)
            X[i] = m.T; ok.append(i)
        except Exception as e:
            n_fail += 1
            if n_fail <= 3: bar.write(f"  decode fail: {os.path.basename(p)}: {e}")
        if (i & 255) == 0:
            bar.set_postfix(ok=len(ok), failed=n_fail)
    bar.close()
    print(f"  decoded {len(ok)}/{len(paths)} files "
          f"({n_fail} failed) in {_time.time()-_t0:.0f}s", flush=True)
    X, Y = X[ok], torch.tensor([labs[i] for i in ok])
    mu, sd = X.mean(), X.std()
    return (X - mu) / (sd + 1e-6), Y

def hd_loaders(batch=None):
    batch = batch or CFG["BATCH"]
    if CFG["USE_SYNTHETIC"]:
        return synth_loaders(CFG["N_MELS"], spike_input=False, batch=batch)
    from sklearn.model_selection import train_test_split
    cache = os.path.join(CFG["DATA_DIR"], f"hd_mel_{CFG['N_MELS']}_{CFG['T']}.pt")
    if os.path.exists(cache):
        X, Y = torch.load(cache); print("loaded HD cache", tuple(X.shape))
    else:
        X, Y = build_hd_cache(kept, labels, CFG["N_MELS"], CFG["T"])
        torch.save((X, Y), cache); print("built + cached HD", tuple(X.shape))
    idx_tr, idx_te = train_test_split(np.arange(len(Y)), test_size=0.2,
                                      random_state=0, stratify=Y.numpy())
    L = lambda idx, s: DataLoader(TensorDS(X[idx], Y[idx]), batch_size=batch, shuffle=s)
    print("HD train/test:", len(idx_tr), "/", len(idx_te))
    return L(idx_tr, True), L(idx_te, False)

print("HD loader ready (synthetic)" if CFG["USE_SYNTHETIC"] else "HD loader ready (real)")

---
## Setup cell 6 — model definitions

All five models share the shape `n_in → HIDDEN → 20` over `T` timesteps, so only the **architecture** varies. The structural difference between the cheap and expensive hybrids is one line of indentation:

- **SNN-front → ANN-back (Section 3):** the readout `fc2` sits **outside** the time loop → runs **once**.
- **ANN-front → SNN-back (Section 4):** the dense `fc1` sits **inside** the time loop → runs **`T` times**.

That single difference is the whole finding. `spike_input=True` tells the counter that layer 1 is driven by binary activations (SHD), not real values (HD).

*(One small correctness fix versus a hand-written earlier draft: spike totals are read with `.detach()` before `float()`, so the counting hook is safe to call even outside `torch.no_grad()`. It changes no count — it only avoids a spurious autograd warning.)*

In [ ]:
import torch.nn as nn                 # nn: layers & losses (nn.Linear, nn.Module, nn.CrossEntropyLoss)
import snntorch as snn                 # snn: spiking neurons (snn.Leaky = LIF)
from snntorch import surrogate         # surrogate gradients for the non-differentiable spike

SG = surrogate.fast_sigmoid()          # the smooth backward-pass stand-in for the hard spike (primer 6)

def make_lif(beta=None):
    """Build a LIF neuron. I use LIF throughout because it is what commercial
    neuromorphic hardware (Intel Loihi) implements as its built-in primitive;
    see the note in section 2.4. beta (<1) is the membrane decay per timestep."""
    # snn.Leaky IS a LIF layer: beta = per-step membrane decay, spike_grad = surrogate (primer 6)
    return snn.Leaky(beta=(CFG["BETA"] if beta is None else beta), spike_grad=SG)

def _spk(t):  # safe spike total for the counter (.detach() = "don't track this for gradients", primer 4)
    return float(t.detach().sum())

class Base(nn.Module):                  # nn.Module = base class for anything with learnable weights (primer 3)
    def __init__(self, n_in, n_hid, n_out, T, neuron, spike_input):
        super().__init__()              # REQUIRED first line for every nn.Module subclass
        self.n_in, self.n_hid, self.n_out = n_in, n_hid, n_out
        self.T, self.neuron, self.spike_input = T, neuron, spike_input
        self.syn1, self.syn2 = n_in*n_hid, n_hid*n_out
    @property
    def n_syn(self):  return self.syn1 + self.syn2
    @property
    def n_neur(self): return self.n_hid + self.n_out

class ModelANN(Base):
    """Pure ANN. Per-frame MLP + temporal mean pooling. fc1 and fc2 both run T times.
    Has NO spiking neurons, so no membrane and no leak; its neuron label is '-'."""
    def __init__(self, n_in, n_hid, n_out, T, neuron="-", spike_input=False):
        super().__init__(n_in, n_hid, n_out, T, neuron, spike_input)
        # nn.Linear(a, b) = a learnable  W@x + b  from a inputs to b outputs (primer 2)
        self.fc1, self.fc2 = nn.Linear(n_in, n_hid), nn.Linear(n_hid, n_out)
    def forward(self, x, stats=None):   # called via model(x), never directly (primer 3)
        out = 0
        for t in range(x.shape[1]):     # x is [batch, T, features]; loop over the T time frames
            xt = x[:, t]                # timestep t for the whole batch: shape [batch, features]
            h = torch.relu(self.fc1(xt))          # dense layer + ReLU (torch.relu == np.maximum(0, .))
            out = out + self.fc2(h)               # accumulate the logits across time
        if stats is not None:
            B = x.shape[0]; stats.mac_syn += B * self.T * (self.syn1 + self.syn2)
        return out / x.shape[1]         # mean-pool over time

class ModelSNN(Base):
    """Pure SNN. Layer 1 = ACs if input is spikes, else MACs."""
    def __init__(self, n_in, n_hid, n_out, T, neuron="LIF", spike_input=True):
        super().__init__(n_in, n_hid, n_out, T, neuron, spike_input)
        self.fc1, self.fc2 = nn.Linear(n_in, n_hid), nn.Linear(n_hid, n_out)
        self.lif1, self.lif2 = make_lif(), make_lif()   # two LIF layers (each has its own membrane)
    def forward(self, x, stats=None):
        # init_leaky() creates each neuron's starting membrane potential (zeros) -- primer 6
        m1, m2 = self.lif1.init_leaky(), self.lif2.init_leaky()
        out, in_spk, h_spk = 0, 0.0, 0.0
        for t in range(x.shape[1]):
            xt = x[:, t]
            # THE core snnTorch call: (input current, prev membrane) -> (spikes, new membrane)
            s1, m1 = self.lif1(self.fc1(xt), m1)   # layer 1: weights -> LIF -> spikes s1
            s2, m2 = self.lif2(self.fc2(s1), m2)   # layer 2: driven by layer-1 spikes
            out = out + s2                          # accumulate output spikes over time
            in_spk += _spk(xt); h_spk += _spk(s1)   # tally spike counts for the energy model
        if stats is not None:
            B = x.shape[0]
            if self.spike_input: stats.ac_syn  += in_spk * self.n_hid
            else:                stats.mac_syn += B * self.T * self.syn1
            stats.ac_syn += h_spk * self.n_out
            # both layers are LIF -> both leak every timestep, over ALL neurons
            stats.lif_steps += B * self.T * self.n_neur
            stats.spikes["L1"] = stats.spikes.get("L1", 0.) + in_spk / max(self.n_in, 1)
            stats.spikes["L2"] = stats.spikes.get("L2", 0.) + h_spk / max(self.n_hid, 1)
        return out / x.shape[1]

class ModelSNNfront(Base):
    """SNN front -> accumulator -> ANN readout. fc2 is OUTSIDE the loop: runs ONCE."""
    def __init__(self, n_in, n_hid, n_out, T, neuron="LIF", spike_input=True):
        super().__init__(n_in, n_hid, n_out, T, neuron, spike_input)
        self.fc1, self.fc2 = nn.Linear(n_in, n_hid), nn.Linear(n_hid, n_out)
        self.lif1 = make_lif()                          # hidden layer is LIF
    def forward(self, x, stats=None):
        m1 = self.lif1.init_leaky()
        acc, in_spk = 0, 0.0
        for t in range(x.shape[1]):
            xt = x[:, t]
            s1, m1 = self.lif1(self.fc1(xt), m1)   # spiking hidden layer (LIF), inside the time loop
            acc = acc + s1; in_spk += _spk(xt)     # accumulator sums the spikes over time
        acc = acc / x.shape[1]
        if stats is not None:
            B = x.shape[0]
            if self.spike_input: stats.ac_syn  += in_spk * self.n_hid
            else:                stats.mac_syn += B * self.T * self.syn1
            stats.mac_syn += B * self.syn2                    # readout: ONCE (ANN, no membrane)
            # only the hidden LIF leaks; the ANN readout has no membrane
            stats.lif_steps += B * self.T * self.n_hid
            stats.spikes["L1"] = stats.spikes.get("L1", 0.) + in_spk / max(self.n_in, 1)
        return self.fc2(acc)     # <-- readout is OUTSIDE the loop: runs ONCE on the time-collapsed acc

class ModelANNfront(Base):
    """ANN front -> spike encoder -> SNN back.
       collapse=False: fc1 INSIDE the loop -> T x dense MACs (true architecture comparison)
       collapse=True : time averaged first -> fc1 runs once (a different experiment)."""
    def __init__(self, n_in, n_hid, n_out, T, neuron="LIF", spike_input=False, collapse=False):
        super().__init__(n_in, n_hid, n_out, T, neuron, spike_input)
        self.collapse = collapse
        self.fc1, self.fc2 = nn.Linear(n_in, n_hid), nn.Linear(n_hid, n_out)
        self.enc  = make_lif()       # encoder integrates analog -> spikes (LIF)
        self.lif2 = make_lif()       # spiking back-end layer (LIF)
    def forward(self, x, stats=None):
        B, Tt = x.shape[0], x.shape[1]
        me, m2 = self.enc.init_leaky(), self.lif2.init_leaky()
        out, enc_spk = 0, 0.0
        if self.collapse:
            h_static = torch.relu(self.fc1(x.mean(1)))     # ONE dense pass
        for t in range(Tt):
            h = h_static if self.collapse else torch.relu(self.fc1(x[:, t]))
            s, me = self.enc(h, me); enc_spk += _spk(s)     # analog -> binary
            s2, m2 = self.lif2(self.fc2(s), m2); out = out + s2
        if stats is not None:
            stats.mac_syn += B * self.syn1 * (1 if self.collapse else Tt)
            stats.ac_syn  += enc_spk * self.n_out           # encoder spikes -> ACs
            # both the encoder (hidden) and the LIF back-end (output) leak every step
            stats.lif_steps += B * Tt * (self.n_hid + self.n_out)
            stats.spikes["enc"] = stats.spikes.get("enc", 0.) + enc_spk / max(self.n_hid, 1)
        return out / Tt


class ModelSNNRecurrent(Base):
    """STRONGER SNN (the "option 2" model): a RECURRENT spiking hidden layer, then an
    ANN readout after the accumulator.

    WHY THIS EXISTS. The plain feed-forward SNNs above converge to well below the ANN's
    accuracy, which makes a like-for-like energy comparison unfair (you cannot compare
    energy at very different accuracies -- see the iso-accuracy discussion). Published
    SHD results in the 0.80-0.90 range use *recurrent* spiking layers, because speech
    needs memory across timesteps. This model adds exactly that.

    WHAT CHANGES, MECHANICALLY. Each hidden neuron now also receives the previous
    timestep's hidden spikes through a recurrent weight matrix `rec` (n_hid x n_hid).
    Those recurrent synapses are driven by BINARY spikes, so they are ACs, not MACs --
    and they are real extra synapses, so they are added to N_syn and paid for honestly
    in the energy model. Accuracy should go up; energy will go up too. That is the
    point: we re-measure at the higher-accuracy operating point rather than pretending
    the improvement is free.
    """
    def __init__(self, n_in, n_hid, n_out, T, neuron="LIF", spike_input=True):
        super().__init__(n_in, n_hid, n_out, T, neuron, spike_input)
        self.fc1  = nn.Linear(n_in, n_hid)
        self.rec  = nn.Linear(n_hid, n_hid, bias=False)   # recurrent spiking connections
        self.fc2  = nn.Linear(n_hid, n_out)               # ANN readout, AFTER the accumulator
        self.lif1 = make_lif()
        self.syn_rec = n_hid * n_hid                      # extra synapses, counted honestly
    @property
    def n_syn(self):  return self.syn1 + self.syn_rec + self.syn2
    @property
    def n_neur(self): return self.n_hid                   # readout is ANN -> no membrane

    def forward(self, x, stats=None):
        B, Tt = x.shape[0], x.shape[1]
        m1 = self.lif1.init_leaky()
        s1 = torch.zeros(B, self.n_hid, device=x.device, dtype=x.dtype)  # previous spikes
        acc, in_spk, rec_spk = 0, 0.0, 0.0
        for t in range(Tt):
            xt = x[:, t]
            cur = self.fc1(xt) + self.rec(s1)      # feed-forward + recurrent drive
            s1, m1 = self.lif1(cur, m1)
            acc = acc + s1
            in_spk  += _spk(xt)
            rec_spk += _spk(s1)                    # spikes that drive the recurrent matrix
        acc = acc / Tt
        if stats is not None:
            if self.spike_input: stats.ac_syn  += in_spk * self.n_hid   # binary input -> ACs
            else:                stats.mac_syn += B * Tt * self.syn1    # real input -> MACs
            stats.ac_syn  += rec_spk * self.n_hid      # recurrent synapses: spike-driven ACs
            stats.mac_syn += B * self.syn2             # ANN readout: paid ONCE
            stats.lif_steps += B * Tt * self.n_hid     # only the hidden layer has a membrane
            stats.spikes["L1"]  = stats.spikes.get("L1", 0.)  + in_spk  / max(self.n_in, 1)
            stats.spikes["rec"] = stats.spikes.get("rec", 0.) + rec_spk / max(self.n_hid, 1)
        return self.fc2(acc)

print("models defined  (incl. ModelSNNRecurrent = the stronger option-2 SNN)")

---
## Setup cell 7 — structural self-test (runs instantly, no data needed)

Before any training, this cell forwards each model on tiny synthetic tensors and asserts the **structural invariants** from the counting rule. These are the checks that catch the errors that matter — a pure ANN with stray ACs, a readout paid `T` times instead of once, a per-timestep front end that isn't actually `T×` the collapsed one. If any assertion fails, the counting is wrong and nothing downstream can be trusted.

*(This reproduces, against live `snntorch`, the same expectations as the standalone `count_test.py`.)*

In [ ]:
def _count(model, x):
    st = OpStats(n_syn=model.n_syn, n_neur=model.n_neur)
    _ = model(x, stats=st)
    return st

_T, _H, _C, _B = CFG["T"], CFG["HIDDEN"], CFG["N_CLASSES"], 4
_xspk  = (torch.rand(_B, _T, 700) < 0.05).float()   # SHD-like binary
_xreal = torch.randn(_B, _T, 40)                    # HD-like real

# pure ANN: MAC = B*T*(syn1+syn2), zero ACs, zero membrane (no spiking neurons)
s = _count(ModelANN(40, _H, _C, _T, "-", False), _xreal)
assert s.mac_syn == _B*_T*(40*_H + _H*_C) and s.ac_syn == 0 and s.lif_steps == 0
print("PASS  pure ANN: MAC = B*T*(syn1+syn2), ACs = 0, no membrane")

# pure SNN on spikes: zero MACs; LIF leaks over ALL neurons every timestep
s = _count(ModelSNN(700, _H, _C, _T, "LIF", True), _xspk)
assert s.mac_syn == 0
assert s.lif_steps == _B*_T*(_H+_C)
print("PASS  pure SNN: spike input -> 0 MACs; LIF leak = B*T*N_neur (all neurons)")

# SNN-front -> ANN-back: readout paid ONCE (B*syn2); LIF leaks HIDDEN only
# (the ANN readout has no membrane, so the output layer contributes no leak)
s = _count(ModelSNNfront(700, _H, _C, _T, "LIF", True), _xspk)
assert s.mac_syn == _B*(_H*_C)
assert s.lif_steps == _B*_T*_H
print("PASS  SNN-front: readout MAC = B*syn2 (once); LIF leaks hidden only")

# ANN-front per-step vs collapsed: layer-1 MAC ratio == T
sa = _count(ModelANNfront(40, _H, _C, _T, "LIF", False, collapse=False), _xreal)
sb = _count(ModelANNfront(40, _H, _C, _T, "LIF", False, collapse=True),  _xreal)
assert sa.mac_syn == _B*_T*(40*_H) and sb.mac_syn == _B*(40*_H)
assert sa.mac_syn // sb.mac_syn == _T
print(f"PASS  ANN-front: per-step MAC = T x collapsed MAC (ratio = {_T})")
print("\nALL STRUCTURAL INVARIANTS HOLD -> counting is trustworthy.")

---
## Setup cell 8 — training and evaluation harness

A standard training loop. The evaluation pass additionally accumulates `OpStats` from **actual execution traces**, so every operation and spike count is measured, not derived from architecture arithmetic. Under `USE_SYNTHETIC`, epochs are reduced (accuracy is not the point there) but the counts remain exact.

**Operation and spike counts are exact measurements. Joules are modelled.** Keep those two labels apart in anything you write.

In [ ]:
import time, sys
from tqdm.auto import tqdm
# NOTE: pandas is imported later, in the results cell, not here. The training,
# counting, and energy computations do not need it, so a pandas/NumPy ABI issue
# (see Setup cell 1) cannot block the core run.
#
# PROGRESS INDICATORS (v6). The previous harness printed nothing between the
# start of run() and the end of every seed x epoch, so a slow run and a hung
# run looked identical from the outside. Now: a per-epoch bar with live loss,
# a line per finished seed, and a line per finished configuration.

def train_eval(model, tr, te, epochs=None, lr=None, label="", seed_note=""):
    epochs = epochs if epochs is not None else (CFG["SYN_EPOCHS"] if CFG["USE_SYNTHETIC"] else CFG["EPOCHS"])
    lr = lr or CFG["LR"]
    model = model.to(DEV)                                 # move weights to GPU/CPU (primer 3)
    opt = torch.optim.Adam(model.parameters(), lr=lr)     # optimiser gets every learnable tensor (primer 4)
    lossf = nn.CrossEntropyLoss()                         # classification loss (logits + int labels -> scalar)
    t0 = time.time()
    bar = tqdm(range(epochs), desc=f"{label} {seed_note}", unit="ep", file=sys.stdout, leave=False)
    for ep in bar:
        model.train()                                    # training mode
        tot = nb = 0
        for xb, yb in tr:                                # DataLoader yields batches (primer 5)
            xb, yb = xb.to(DEV), yb.to(DEV)
            # the 4-step autograd ritual (primer 4): reset grads -> forward+loss -> backprop -> update
            opt.zero_grad(); loss = lossf(model(xb), yb); loss.backward(); opt.step()
            tot += float(loss); nb += 1
        bar.set_postfix(loss=f"{tot/max(nb,1):.3f}")
    bar.close()
    model.eval()                                         # eval mode (no training-only behaviour)
    correct = n = 0
    stats = OpStats(n_syn=model.n_syn, n_neur=model.n_neur)
    with torch.no_grad():                                # don't build the gradient graph: we're only measuring
        for xb, yb in te:
            xb, yb = xb.to(DEV), yb.to(DEV)
            out = model(xb, stats=stats)                 # forward pass ALSO fills in op-counts via stats
            correct += int((out.argmax(1) == yb).sum()); n += len(yb)  # argmax over classes = prediction
    # ---- BUG FIX (v10) ----------------------------------------------------
    # Do NOT divide by len(te) here. `stats.spikes` accumulates the RAW spike
    # totals over the whole test set (already divided by the presynaptic layer
    # width inside each model). `per_sample(n)` below divides by the number of
    # SAMPLES, which is the correct and only normalisation needed.
    # An earlier version also divided by the batch count, which made every
    # reported spikes/synapse figure len(te)x too small -- about 18x on SHD.
    # The energy columns were never affected (they derive from ac_syn, which is
    # normalised correctly); only the diagnostic spk/syn column was wrong.
    # -----------------------------------------------------------------------
    row = stats.per_sample(n).summary(label)
    row["accuracy"] = round(correct / n, 4)
    row["train_s"]  = round(time.time() - t0, 1)
    return row

RESULTS = []

# ---- checkpointing: RESULTS is written to Drive after every finished configuration,
# ---- so a disconnect never costs you more than the config that was in flight.
def save_results():
    try:
        with open(RESULTS_JSON, "w") as f:
            json.dump(RESULTS, f, indent=1)
    except Exception as e:
        print("  [checkpoint failed]", type(e).__name__, e, flush=True)

def load_results():
    """Restore RESULTS from the last checkpoint (call this after a reconnect)."""
    global RESULTS
    try:
        with open(RESULTS_JSON) as f:
            RESULTS = json.load(f)
        print(f"restored {len(RESULTS)} result rows from {RESULTS_JSON}")
        for r in RESULTS:
            print("   ", r["model"])
    except FileNotFoundError:
        print("no checkpoint yet at", RESULTS_JSON)
    return RESULTS

def run(tag, ctor, tr, te, neurons=None):
    # LIF is the only spiking neuron used (see 2.4). A pure ANN passes neurons=["-"].
    ns = neurons or [CFG["NEURON"]]
    n_ep = CFG["SYN_EPOCHS"] if CFG["USE_SYNTHETIC"] else CFG["EPOCHS"]
    print(f"\n>>> {tag}  |  {len(ns)} neuron(s) x {len(CFG['SEEDS'])} seeds x {n_ep} epochs"
          f"  |  {len(tr)} train batches/epoch on {DEV}", flush=True)
    t_cfg = time.time()
    for neuron in ns:
        accs = []
        for si, seed in enumerate(CFG["SEEDS"], 1):
            set_seed(seed)
            t_seed = time.time()
            r = train_eval(ctor(neuron), tr, te, label=f"{tag} [{neuron}]",
                           seed_note=f"seed {si}/{len(CFG['SEEDS'])}")
            accs.append(r["accuracy"])
            print(f"    seed {seed}  acc {r['accuracy']:.3f}  ({time.time()-t_seed:.0f}s)", flush=True)
        r["accuracy_mean"] = round(float(np.mean(accs)), 4)
        r["accuracy_std"]  = round(float(np.std(accs)), 4)
        r["neuron"] = neuron
        RESULTS.append(r)
        save_results()                      # <-- checkpoint to Drive after every config
        print(f"{tag:34s} {neuron:3s}  acc {r['accuracy_mean']:.3f}±{r['accuracy_std']:.3f}"
              f"  MAC {r['MACs']:>9,}  AC {r['ACs']:>9,}  E_mem {r['E_memory_norm']:>12,.0f}", flush=True)
    print(f"<<< {tag} done in {time.time()-t_cfg:.0f}s", flush=True)
    return RESULTS

print("harness ready  |  results checkpoint ->", RESULTS_JSON)
print("harness ready  |  mode:", "SYNTHETIC (counts real, accuracy ~chance)" if CFG["USE_SYNTHETIC"] else "REAL DATA")


---
# Section 1 — HD spectrogram → pure ANN  *(my honest baseline)*

**Input:** raw HD audio → log-mel spectrogram → real-valued `[T, 40]` tensor.
**Architecture:** a dense MLP applied per frame, then averaged over time. Both layers run every timestep.

This is the conventional audio classifier, and it's the baseline everything else is measured against. Two structural facts I want you to watch, both predicted by §2.7 and §2.2:

- It has **zero ACs**. Real-valued input means every synapse is a MAC. (My own sanity check: if this row ever shows ACs, I miscounted something.)
- It has **no spiking neurons at all**, so it has no membrane and pays no leak. I label its neuron column `—` rather than forcing "LIF" onto a network that has none.
- Both layers run `T = 25` times, so its MAC count is `T × (syn1 + syn2)` — the price of processing a time axis densely.


In [ ]:
hd_tr, hd_te = hd_loaders()
run("S1 HD -> ANN",
    lambda nu: ModelANN(CFG["N_MELS"], CFG["HIDDEN"], CFG["N_CLASSES"], CFG["T"],
                        neuron=nu, spike_input=False),
    hd_tr, hd_te, neurons=["-"])   # a pure ANN has no spiking neuron; label it "-"

**Section 1 — what I take from this.**

The pure ANN sets the accuracy bar the sparse models have to clear, at a MAC count of `T×(syn1+syn2)` and **exactly zero ACs** — which confirms §2.7's claim that a real-valued front end is unavoidably multiply-based. In the memory-inclusive model each of those MAC-synapses costs **22.6** normalised units (§2.2), almost all of it memory traffic, not the multiply. That's the number the SNN's per-spike cost of **16.33** has to beat *often enough* — i.e. while firing fewer than the break-even spikes/synapse of §2.3. I keep this row's `E_memory_norm` in view: it's the denominator for every "×better / ×worse" statement I make later.


---
# Section 2 — SHD spikes → pure SNN  *(genuinely event-native input)*

**Input:** SHD cochlea spikes, binarised to `[T, 700]` of 0/1.
**Architecture:** two LIF spiking layers.

This is the one configuration whose **first layer is genuinely spike-driven**, so layer 1's synapses are **ACs**, not MACs — the structural advantage §2.7 says a spectrogram can never have. Two things I'm watching:

- **Whether the leak earns its cost.** Every LIF neuron pays `N_neur × T × 11.8` for the leak (§2.4), spike or not. At SHD's native ratio of 623 (§2.5) that leak is only ~5.5% of total, so it should be a small share here — but it's real, and it's the reason Section 3 will look cheaper.
- **Whether the measured `N_spikes/syn` actually lands below the break-even targets of §2.3.** If the trained network fires too much, its "efficiency" evaporates — break-even is a real threshold, not a formality. I genuinely don't know the answer until I run on real SHD; the synthetic run can't tell me.

> **Caveat I carry forward:** the cochlea model that turned audio into these spikes costs real energy that *no* model here counts. "SNN wins on audio" quietly assumes that transduction is free. Section 1's FFT front end is uncounted too — neither pipeline's front end is in my ledger.


In [ ]:
shd_tr, shd_te = shd_loaders(CFG["SHD_CHANNELS"])
run("S2 SHD -> SNN",
    lambda nu: ModelSNN(CFG["SHD_CHANNELS"], CFG["HIDDEN"], CFG["N_CLASSES"], CFG["T"],
                        neuron=nu, spike_input=True),
    shd_tr, shd_te)

**Section 2 — what I take from this.**

Because the input is binary, layer 1 contributes **ACs, not MACs** (my check: this row's MAC count should be ~0). That's the event-native advantage in its purest form. Whether this pure SNN actually beats the Section-1 ANN depends entirely on the measured `N_spikes/syn` against §2.3's thresholds — I read that column rather than assume a win. And since I'm using LIF, this network pays the leak over *all* its neurons; that cost is exactly what Section 3 removes from the output layer by handing the readout to an ANN.


---
# Section 3 — SHD spikes → SNN front → ANN back  *(the cheap hybrid)*

**Input:** SHD spikes.
**Architecture:** a LIF spiking layer processes the event stream over `T` steps; an **accumulator collapses time**; a dense ANN readout runs **once**.

This is the mechanically favourable hybrid, and the reason is one line of code: the readout sits **outside** the time loop. Because the accumulator has already summed over time, the `HIDDEN × 20` readout synapses are paid **once**, not `T` times. Compare this row's MAC count to Section 2's — the readout MACs should be a small one-off, not a per-timestep tax.

There's a second saving that only shows up because I use LIF: **the ANN readout has no membrane, so it pays no leak.** Section 2's SNN leaks over all 148 neurons; this hybrid leaks over the 128 hidden neurons only. Putting the output layer in ANN-land removes its membrane cost entirely.

**My prediction before running (from §2.2 + the once-vs-`T` asymmetry + the removed output-layer leak):** this should be among the cheapest configurations. It's the concrete form of Dampfhoffer's "SNN where sparse, ANN where dense" rule.


In [ ]:
run("S3 SHD -> SNNfront+ANNback",
    lambda nu: ModelSNNfront(CFG["SHD_CHANNELS"], CFG["HIDDEN"], CFG["N_CLASSES"], CFG["T"],
                             neuron=nu, spike_input=True),
    shd_tr, shd_te)

**Section 3 — what I take from this.**

The readout's MAC cost appears **once** (`B × syn2`), not `T` times — that's the whole mechanical point, and my structural test asserts it fires exactly once. And because I use LIF, the readout-as-ANN also removes the output layer's membrane traffic: this variant leaks over the **hidden layer only** (`B×T×hidden`), cheaper than Section 2's leak over all neurons. This is the "collapse time with an accumulator, then read out cheaply" pattern that the 2021 Hybrid SNN-ANN DenseNet **[ABSTRACT]** and Dampfhoffer's hybrid rule both point to. If the energy-model ranking puts this configuration near the cheapest, that's my central positive result: **placing the boundary so the dense part runs once — and the output layer has no membrane — is where the savings come from.**


---
# Section 4 — HD spectrogram → ANN front → SNN back  *(the expensive hybrid, and why)*

**Input:** HD mel-spectrogram (real-valued).
**Architecture:** a dense ANN front end, a spike encoder, then a LIF spiking back end. Two variants:

- **4a, per-timestep:** the dense `fc1` runs **inside** the time loop → `T ×` dense MACs. This is the true architecture comparison against Section 3.
- **4b, collapsed:** frames are averaged first, so `fc1` runs **once**. Cheap — but it discards temporal structure, making it a *different experiment* (a plain spectrogram classifier with a spiking tail), not a like-for-like hybrid.

**My prediction (from §2.2 and the run-once vs run-`T` asymmetry):** 4a should cost roughly `T ×` more layer-1 MACs than 4b, and should be clearly worse than Section 3, because a *dense* front end on a time series can't exploit sparsity and pays the full `T` tax. This is the mechanical mirror image of Section 3, and it's the core of my "boundary placement matters" argument: same two ingredients, opposite order, very different cost.


In [ ]:
run("S4a HD -> ANNfront(per-step)+SNN",
    lambda nu: ModelANNfront(CFG["N_MELS"], CFG["HIDDEN"], CFG["N_CLASSES"], CFG["T"],
                             neuron=nu, spike_input=False, collapse=False),
    hd_tr, hd_te)
run("S4b HD -> ANNfront(collapsed)+SNN",
    lambda nu: ModelANNfront(CFG["N_MELS"], CFG["HIDDEN"], CFG["N_CLASSES"], CFG["T"],
                             neuron=nu, spike_input=False, collapse=True),
    hd_tr, hd_te)

**Section 4 — what I take from this.**

4a's layer-1 MAC count should be **exactly `T ×` 4b's** (my counting test asserts this ratio = 25) — a dense front end on a time axis pays for every timestep. That's why an ANN-front hybrid is the expensive way to place the boundary: the dense half lands where the data is still `T`-deep, so it runs `T` times — precisely the tax Section 3 avoids by putting the dense half *after* the accumulator. 4b dodges the tax only by throwing away time, which changes the task. Together, Sections 3 and 4 are my thesis in miniature: **the same ANN and SNN pieces cost more or less depending only on which side of the accumulator the dense layer sits.** The realistic prize from doing this well is roughly 1.2–2.45× (Louis et al. **[ABSTRACT]**), not the orders of magnitude vendor comparisons imply.


---
# Section 5 — Matched-dimension control (both datasets at 40 channels)

Sections 1–4 use each dataset at its natural width — SHD at 700 channels, HD at 40 mel bins. That makes any *cross-dataset* energy comparison unfair, because synapse count scales with input width: SHD simply has more synapses to begin with.

This control downsamples SHD to 40 channels so both share an input dimension. It costs SHD dearly on purpose: `N_syn/N_neur` collapses from **623 to 52** — dropping it into the regime where §2.5 says the LIF leak term dominates (that table's ~41%-leak row).

**Use Sections 1–4 for architecture conclusions within a dataset. Use Section 5, and only Section 5, for any cross-dataset claim.** This separation is my main guard against the most common apples-to-oranges error in this literature.


In [ ]:
m_tr, m_te = shd_loaders(CFG["MATCHED_CH"])
print(f"matched SHD: {CFG['MATCHED_CH']} channels")
run(f"S5 SHD@{CFG['MATCHED_CH']} -> SNN",
    lambda nu: ModelSNN(CFG["MATCHED_CH"], CFG["HIDDEN"], CFG["N_CLASSES"], CFG["T"],
                        neuron=nu, spike_input=True),
    m_tr, m_te)
run(f"S5 SHD@{CFG['MATCHED_CH']} -> SNNfront+ANN",
    lambda nu: ModelSNNfront(CFG["MATCHED_CH"], CFG["HIDDEN"], CFG["N_CLASSES"], CFG["T"],
                             neuron=nu, spike_input=True),
    m_tr, m_te)

---
# Section 5b · Closing the accuracy gap: a stronger (recurrent) SNN

**Why this section exists.** Sections 2, 3 and 5 gave the spiking models accuracies well below the pure ANN. That is a genuine problem for the comparison: **energy is only meaningful at comparable accuracy.** A model that is cheap because it is bad has not won anything — taken to the extreme, a network that always guesses "3" costs almost nothing and is useless.

**What changed, and what did not.** The only change is a **recurrent** spiking hidden layer: each hidden neuron now also sees the previous timestep's hidden spikes. Speech is a temporal signal and needs memory across timesteps, which a plain feed-forward SNN does not have; published SHD results in the 0.80–0.90 range use recurrence for exactly this reason. The energy model is **unchanged**, and the extra recurrent synapses (`n_hid × n_hid`, spike-driven, so ACs) are **counted honestly** and added to `N_syn`.

**What to expect, stated before running so it cannot be rationalised afterwards.** Accuracy should rise substantially. Energy should *also* rise, because recurrence adds real synapses. The question this section answers is therefore the right one:

> At an accuracy much closer to the ANN's, is the spiking model still competitive on energy — or did we simply buy accuracy with energy?

> **A note on what training longer can and cannot fix.** Training for more epochs changes only the *weights*, not the per-inference operation counts, so it moves accuracy without changing energy. But the accuracy ceiling of the plain feed-forward SNN is an **architecture** limit, not an undertraining one (its 3-seed error bars are tight, i.e. it has converged). Fixing an architecture limit requires architecture changes — recurrence, width — and those *do* cost energy. That is why this section re-measures rather than assuming.

In [ ]:
# Stronger recurrent SNN on SHD, at native 700 channels (same data as S2/S3).
run("S6 SHD -> RecurrentSNN+ANNback",
    lambda nu: ModelSNNRecurrent(CFG["SHD_CHANNELS"], CFG["HIDDEN"], CFG["N_CLASSES"],
                                 CFG["T"], neuron=nu, spike_input=True),
    shd_tr, shd_te)

**Section 5b — how to read this.**

Compare the `S6` row against **`S2` (plain SNN)** and **`S1` (the ANN baseline)** on three columns together — never on energy alone:

1. **Accuracy** — did recurrence actually close most of the gap to the ANN? If S6 is near the ANN, the energy comparison finally becomes a fair one.
2. **`N_syn` and ACs** — recurrence adds `HIDDEN²` synapses (16,384 at HIDDEN=128) and they are spike-driven, so they show up as extra ACs. This is the price of the accuracy.
3. **`E_memory_norm` per unit of accuracy** — the honest question. If S6 reaches ANN-like accuracy at lower energy than S1, that is a real result. If it reaches ANN-like accuracy only by spending *more* energy than S1, that is also a real result, and a more interesting one to report than a win.

Whatever the numbers say, report the accuracy alongside every energy claim from here on.

**Section 5 — what I take from this.**

At 40 channels the leak's share of energy should jump from ~5.5% (Section 2, native SHD) to ~41%, because the ratio falls from 623 to 52 (§2.5). If that jump shows up, it directly confirms the mechanism: **the affordability of a leaky neuron is a property of the architecture's synapses-per-neuron ratio, not of the dataset or the task.** It's also a cautionary tale about a routine "speed" optimisation — downsampling SHD, done casually, silently makes the leak dominate the energy budget. I read cross-dataset energy numbers *here*, where input width is matched, and nowhere else.


---
## Compute budget — what each section costs, and what you may safely skip

Colab's free tier gives a limited amount of GPU time per day. These are the measured costs of a full audio run (3 seeds × 20 epochs, T4 GPU), so you can decide what to spend it on.

| Section | Time | What you lose by skipping it |
|---|---|---|
| Data prep (HD download + decode, SHD frames) | ~2 min once | everything — but it is cached, so you pay it once |
| **S1** HD → ANN | 42 s | the baseline every other number is relative to. **Essential.** |
| **S2** SHD → SNN | 3.5 min | the event-native measurement. **Essential.** |
| **S3** SHD → SNNfront+ANN | 2 min | half of the boundary-placement result. **Essential.** |
| **S4a** HD → ANNfront(per-step) | 3 min | the other half — the `T`× tax. **Essential.** |
| **S4b** HD → ANNfront(collapsed) | 2.7 min | the Pareto winner. **Essential.** |
| **S5** SHD@40 → SNN | 3 min | the matched-width control — the only thing licensing a cross-dataset claim. **Essential.** |
| S5 SHD@40 → SNNfront+ANN | 1.7 min | a repeat of the S2→S3 comparison at 40 channels. **Safe to skip** if short on time. |
| **S6** recurrent SNN | 2.5 min | the accuracy-gap answer. Keep if you can. |
| **Section 7** (vision) | ~1 GB download + 15–25 min | the cross-domain check. Skip with `RUN_VISION = False` until the audio half is finished. |

**Total for the audio half: about 20 minutes** plus a one-time ~2 minutes of data preparation.

### Three switches for a cheaper run

- **`QUICK = True`** — 2 seeds × 10 epochs instead of 3 × 20. Roughly **2× faster**. Error bars get wider, so use it to check the pipeline, then do the final run with `QUICK = False`.
- **`RUN_VISION = False`** — skips Section 7 entirely, avoiding the ~1 GB N-MNIST download and the longest training runs.
- **`NMNIST_TRAIN` / `NMNIST_TEST`** — default 10,000 / 2,000. The full 60,000-sample set would build a **3.5 GB** tensor and exhaust Colab's RAM. Operation counts are per-inference and completely unaffected by how many samples you use, so subsampling costs you nothing except a little accuracy precision.

> **If you run out of GPU quota mid-way:** nothing is lost. Every finished configuration is checkpointed to Drive, and the dataset caches persist. When your quota resets, run the setup cells and then `load_results()` to restore everything, and continue from where you stopped.

---
# Section 6 — Results: comparing all sections

> **Reconnected after a disconnect?** You do not need to re-train anything you already finished. Run the setup cells (install → config → Drive → energy models → counting → loaders → models → harness), then run:
>
> ```python
> load_results()      # restores every completed configuration from Drive
> ```
>
> and continue from whichever section you had not reached. The dataset caches on Drive load in seconds, and `RESULTS` is restored from the checkpoint written after each configuration.

Three views, in increasing order of what they let me claim:

1. **Full table** — every measurement, all models.
2. **Energy-model agreement** — does the cheap→expensive ranking survive swapping the energy constants? Agreement means the ordering is robust; disagreement is a finding, and the memory-inclusive model wins the tie on digital hardware.
3. **Accuracy vs energy** — the Pareto view: which configurations buy the most accuracy per unit of (modelled) energy.

Everything below is assembled from the exact operation traces measured during evaluation, then costed under both energy models. I read the two energy columns together and never quote one alone.

> ### Read energy and accuracy together — always
>
> The single easiest way to draw a wrong conclusion from the table below is to sort by energy and declare a winner. **Energy is only meaningful at comparable accuracy.** A model can be cheap because it is bad; in the limit, a network that always guesses the same class costs almost nothing and is worthless.
>
> So, when reading these results:
> - Compare architectures **within** a dataset for architecture claims (Sections 1–4, or the vision runs), because those share a task and an input width.
> - Use **Section 5** (matched channels) for anything cross-dataset — the others differ in input width, which alone changes synapse counts.
> - Use **Section 5b / 7.2b** (the recurrent SNN) when you want an energy comparison at accuracy closer to the ANN's — that is the closest this notebook gets to a true iso-accuracy comparison.
> - Quote the **accuracy alongside every energy number**, and prefer the accuracy-vs-energy scatter (§6.3) over any single ranking. The honest output of this project is a **Pareto frontier**, not a winner.

> ### How to read the `spk/syn` column
>
> It is **spikes per synapse per inference** — how many times the average connection is driven during one classification. Two things to check against it:
>
> - **Compare it to `T × 1.38 = 34.5`, not to 1.38** (§2.3). Under that number, the SNN is doing less synaptic work than the `T`-pass ANN *on the same topology*.
> - **It cannot settle a cross-dataset comparison.** SHD-700 has 12× the synapses of HD-40, so a low spike rate there can still mean more total operations. For anything cross-dataset use Section 5, where the widths match.
>
> *(Fixed in v10: this column was previously divided by the batch count as well as the sample count, making it ~18× too small. If you are looking at results generated before this fix, multiply the old `spk/syn` values by the number of test batches.)*

### 6.1 · Full results table

In [ ]:
import pandas as pd
df = pd.DataFrame(RESULTS)
cols = ["model","neuron","accuracy_mean","accuracy_std","MACs","ACs","lif_steps",
        "N_syn","N_neur","syn_per_neur","E_opcount_fp32_nJ","E_opcount_int32_nJ",
        "E_memory_norm","train_s"]
cols += [c for c in df.columns if c.startswith("spk/syn")]
df = df[[c for c in cols if c in df.columns]]
pd.set_option("display.width", 250, "display.max_columns", 60)
try:
    from IPython.display import display; display(df)
except Exception:
    print(df.to_string(index=False))
df.to_csv("results.csv", index=False)
print("saved results.csv")

### 6.2 · Do the two energy models agree on the ranking?

If op-count (a) and memory-inclusive (b) produce the same cheap→expensive ordering, the ranking is robust to the constant choice. If they diverge, that is a finding, and per §5 the memory-inclusive model is the defensible one on digital hardware.

In [ ]:
r_op  = df.sort_values("E_opcount_fp32_nJ")["model"].tolist()
r_mem = df.sort_values("E_memory_norm")["model"].tolist()
print("cheapest -> most expensive\n")
print("op-count (fp32):")
for i, m in enumerate(r_op, 1):  print(f"  {i}. {m}")
print("\nmemory-inclusive:")
for i, m in enumerate(r_mem, 1): print(f"  {i}. {m}")
print("\nRANKINGS AGREE" if r_op == r_mem else
      "\n*** RANKINGS DISAGREE -> report both; memory-inclusive is the defensible one ***")

### 6.3 · Accuracy vs energy, and the operation mix

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(19, 5))

d = df.sort_values("E_memory_norm")
ax[0].barh(d["model"] + " " + d["neuron"], d["E_memory_norm"], color="#3D6489")
ax[0].set_xscale("log"); ax[0].set_xlabel("energy (normalised units, log)")
ax[0].set_title("Memory-inclusive energy (Model b)")

ax[1].scatter(df["E_memory_norm"], df["accuracy_mean"], s=70, color="#E8A33D")
for _, r in df.iterrows():
    ax[1].annotate(r["model"].split()[0] + r["neuron"],
                   (r["E_memory_norm"], r["accuracy_mean"]), fontsize=7)
ax[1].set_xscale("log"); ax[1].set_xlabel("energy (normalised, log)")
ax[1].set_ylabel("test accuracy"); ax[1].set_title("Accuracy vs energy")

w = 0.38; y = np.arange(len(df))
ax[2].barh(y - w/2, df["MACs"], w, label="MACs", color="#E24B4A")
ax[2].barh(y + w/2, df["ACs"],  w, label="ACs",  color="#1D9E75")
ax[2].set_yticks(y); ax[2].set_yticklabels(df["model"] + " " + df["neuron"], fontsize=7)
ax[2].set_xscale("symlog"); ax[2].legend(); ax[2].set_title("Operation mix (MAC vs AC)")

plt.tight_layout(); plt.savefig("results.png", dpi=150); plt.show()
print("saved results.png")

### 6.4 · Every result under **both** ANN accelerator models — and the comparison ledger

Two things happen in this cell, and neither needs a re-run: every measured configuration is
re-priced against the ideal-reuse ANN from Setup 3b, and the ledger from §2.3b is printed from
the actual results so it cannot go stale.

In [ ]:
# Re-price every measured configuration under the IDEAL-REUSE ANN, then print the
# comparison ledger: which axes are actually matched in each headline ratio.
import pandas as pd

TOPO = {  # model-name prefix -> the FC layers whose MACs are being priced
    "S1": [(40,128),(128,20)],   "S2": [(700,128),(128,20)],
    "S3": [(128,20)],            "S4a":[(40,128)],   "S4b":[(40,128)],
    "S5 SHD@40 -> SNN":  [(40,128),(128,20)],
    "S5 SHD@40 -> SNNf": [(128,20)],
    "S6": [(128,20)],
    "V1": [(784,128),(128,10)],  "V2": [(2312,128),(128,10)],
    "V3": [(128,10)],            "V4 ":[(784,128)],  "V4b":[(784,128)],
    "V5": [(128,10)],
}
def layers_for(name):
    for k in sorted(TOPO, key=len, reverse=True):
        if name.startswith(k):
            return TOPO[k]
    return [(128,20)]

rows = []
for r in RESULTS:
    ly      = layers_for(r["model"])
    e_naive = r["E_memory_norm"]
    e_ideal = energy_reuse(r["MACs"], r["ACs"], r["lif_steps"], ly)
    rows.append(dict(model=r["model"], acc=r["accuracy_mean"],
                     spk_per_syn=round(r["ACs"] / r["N_syn"], 3),
                     E_naiveANN=round(e_naive), E_idealANN=round(e_ideal),
                     leak_pct=round(r["lif_steps"] * LIF_LEAK / e_naive * 100, 2)))
d2 = pd.DataFrame(rows)
try:
    from IPython.display import display; display(d2)
except Exception:
    print(d2.to_string(index=False))
d2.to_csv("results_both_ann_models.csv", index=False)
print("saved results_both_ann_models.csv")

print("\n--- COMPARISON LEDGER: what is actually matched in each headline ratio ---")
led = [
 ("S4a vs S4b   boundary placement (audio)", "same", "same", "0.856 vs 0.866 - indistinguishable", "CLEAN"),
 ("V4  vs V4b   boundary placement (vision)","same", "same", "0.946 vs 0.952 - indistinguishable", "CLEAN"),
 ("S2  vs S3    hybrid dominates (audio)",   "same", "same", "hybrid ALSO +0.097",                 "DOMINANCE"),
 ("V2  vs V3    hybrid dominates (vision)",  "same", "same", "equal within noise",                 "DOMINANCE"),
 ("S5  pair     hybrid dominates @40ch",     "same", "same", "hybrid ALSO +0.122",                 "DOMINANCE"),
 ("S1  vs S4b   Pareto (audio)",             "same", "same", "0.966 vs 0.866 - 0.10 apart",        "accuracy gap"),
 ("S1  vs S5    matched-width control",      "DIFF", "same", "0.966 vs 0.316 - 0.65 apart",        "width ok, accuracy NOT"),
 ("S1  vs S2    the old audio headline",     "DIFF", "DIFF", "0.966 vs 0.570",                     "CONFOUNDED x3"),
 ("V1  vs V2    the vision headline",        "DIFF", "DIFF", "0.976 vs 0.9995",                    "CONFOUNDED - see 7.6"),
]
print(f"{'comparison':<42s}{'dataset':>9s}{'width':>7s}  {'accuracy':<36s}verdict")
print("-" * 118)
for c_, ds_, w_, a_, v_ in led:
    print(f"{c_:<42s}{ds_:>9s}{w_:>7s}  {a_:<36s}{v_}")

**How to read the ledger.** Only the two rows marked `CLEAN` are fully controlled **and**
iso-accuracy. The three `DOMINANCE` rows need no accuracy caveat at all, because the cheaper model
is also the equal-or-better one — there is no tradeoff to adjudicate. Everything below the line
carries a confound that must be quoted in the same breath as the number.

**What the better ANN does to the headline claims:**

| Claim | vs **naive** ANN | vs **ideal-reuse** ANN |
|---|---|---|
| Boundary placement, audio — **S4a / S4b** | 7.09× | **4.33×** |
| Boundary placement, vision — **V4 / V4b** | 23.34× | **21.56×** |
| Pareto winner, audio — **S1 / S4b** | 9.18× | **4.82×** |

Boundary placement survives the correction best, because both sides of that comparison are the
same layer type on the same data, so the reuse factor scales them together. **That is a further
argument for making boundary placement — not any cross-dataset number — the headline of this
project.**

---
# Section 7 · A second dataset pair: MNIST and N-MNIST (vision)

*Everything up to here used one dataset pair — HD (audio as numbers) and SHD (audio as spikes). This section repeats the core comparison on a completely different pair, handwritten digits, and then asks the real question: do the conclusions hold, or does the nature of the signal change them?*

**Why add a second pair.** A single dataset can mislead. If the project's claims ("boundary placement is the lever", "the event-native advantage is real but narrow", "the leak's cost tracks N_syn/N_neur") only hold on audio, they aren't claims about hybrids — they're claims about SHD. Vision is the standard second testbed, and it has a matched spiking counterpart, so it is the natural way to check.

**The pair, and how it maps onto the project's two "input regimes":**

- **MNIST** — the famous 28×28 grayscale handwritten digits (10 classes). The pixels are **real-valued numbers**, so this is the "arrives as numbers" case, exactly like HD. Its first layer must be MACs (§2.7). Because a digit image is **static**, feeding it to a spiking network means repeating the *same* image for all T timesteps — this is precisely the "ANN-front → convert-to-spikes → SNN-back" pattern discussed in the handoff document, now made concrete.
- **N-MNIST** — the *neuromorphic* version: each MNIST digit was recorded by an **event camera** that moves ("saccades") across the image, emitting spike events at 34×34 pixels with 2 polarity channels (2312 input channels flattened). This is the "arrives as spikes" case, like SHD. Its first layer can honestly be ACs.

> **The honest asymmetry to keep in front of you (from the handoff doc).** SHD's spikes encode a signal that is *genuinely temporal* — speech unfolds in time. N-MNIST's spikes encode a *static* digit; the only reason there is a time axis at all is the camera's motion. So we should **expect** the SNN's temporal machinery to earn its keep less clearly on N-MNIST than on SHD. If that expectation shows up in the numbers, it is not a disappointment — it is the finding: *whether spiking pays off depends on whether the signal is really temporal, not just on the architecture.*

**Method note — everything is reused.** The five model classes, both energy models, the counting rules, and the training harness are all exactly the ones used above. Only the data and the class count (10, not 20) change. That is deliberate: reusing the identical machinery is what makes the two dataset pairs comparable.

> **Downloads.** N-MNIST via `tonic` is large (~1 GB extracted) and slower to fetch than SHD; the first run of its cell will take several minutes. MNIST via `torchvision` is tiny. Both are cached after the first build. Under `USE_SYNTHETIC = True`, both fall back to random tensors of the right shape (counts/energy real, accuracy at chance), so the section still runs with no downloads.

### Setup 7.1 — MNIST and N-MNIST loaders

Same design as the SHD loader: decode once into an in-memory tensor, cache to disk, train from RAM.

- **MNIST** → each image is flattened to 784 real-valued features and **repeated across T timesteps** (a static image has no time axis of its own). This is the honest way to present a static image to a T-step network, and it is what makes the "dense front end pays T×" cost visible.
- **N-MNIST** → binned to `[T, 2312]` binary frames with the same `ToFrame`-then-binarise recipe as SHD, so layer 1 is genuine ACs. Optionally downsampled for the matched-dimension control.

In [ ]:
# ---------------- MNIST (real-valued, static image -> repeated over T) ----------------
def mnist_cached(T):
    """MNIST as [N, T, 784] float32 (image repeated over T timesteps). Cached."""
    cache = os.path.join(CFG["DATA_DIR"], f"mnist_{T}.pt")
    if os.path.exists(cache):
        d = torch.load(cache)
        print(f"  [MNIST cache hit] train {tuple(d['Xtr'].shape)} test {tuple(d['Xte'].shape)}", flush=True)
        return d["Xtr"], d["Ytr"], d["Xte"], d["Yte"]
    from torchvision import datasets, transforms
    tf = transforms.Compose([transforms.ToTensor()])   # [1,28,28] in [0,1]
    tr = datasets.MNIST(CFG["DATA_DIR"], train=True,  download=True, transform=tf)
    te = datasets.MNIST(CFG["DATA_DIR"], train=False, download=True, transform=tf)
    def to_tensor(ds):
        X = torch.stack([x.view(-1) for x, _ in ds])            # [N, 784]
        Y = torch.tensor([y for _, y in ds], dtype=torch.long)  # [N]
        return X, Y
    Xtr, Ytr = to_tensor(tr); Xte, Yte = to_tensor(te)
    # Normalise and store FLAT, as [N, 784].
    # We deliberately do NOT materialise the T-fold repeat here. Doing so
    # ( .repeat(1,T,1) ) would build a 60000 x 25 x 784 float32 tensor = 4.7 GB,
    # which exhausts Colab's RAM and produces a 4.7 GB cache file. The repeat is
    # applied per-sample in MNISTMem below, costing ~10 MB per batch instead.
    mu, sd = Xtr.mean(), Xtr.std()
    Xtr = ((Xtr - mu) / (sd + 1e-6)).contiguous()
    Xte = ((Xte - mu) / (sd + 1e-6)).contiguous()
    torch.save({"Xtr": Xtr, "Ytr": Ytr, "Xte": Xte, "Yte": Yte}, cache)
    print(f"  [MNIST cached] train {tuple(Xtr.shape)} test {tuple(Xte.shape)}", flush=True)
    return Xtr, Ytr, Xte, Yte

class MNISTMem(Dataset):
    """Holds each image once as [784] and expands it to [T, 784] on access.
    A static image has no time axis of its own, so every timestep sees the same
    frame -- but we generate that view per sample rather than storing T copies."""
    def __init__(self, X, Y, T): self.X, self.Y, self.T = X, Y, T
    def __len__(self):  return len(self.Y)
    def __getitem__(self, i):
        return self.X[i].unsqueeze(0).expand(self.T, -1), int(self.Y[i])

# ---------------- N-MNIST (event camera -> binary frames, like SHD) ----------------
def _build_nmnist_frames(ds, T, n_channels, sensor, desc="", limit=None):
    """Frame + binarise a tonic NMNIST dataset into X uint8 [N, T, C], Y int64 [N].
    C = n_channels (<= 2312). If n_channels < full, we keep the first n_channels
    flattened channels (a simple, reproducible reduction for the matched control)."""
    import tonic.transforms as TT
    from tqdm.auto import tqdm
    full_C = sensor[0] * sensor[1] * sensor[2]      # 34*34*2 = 2312
    tf = TT.Compose([TT.ToFrame(sensor_size=sensor, n_time_bins=T)])
    N = len(ds) if limit is None else min(limit, len(ds))
    out = np.zeros((N, T, n_channels), dtype=np.uint8)
    Y   = np.zeros(N, dtype=np.int64)
    ds.transform = tf
    for i in tqdm(range(N), desc=desc, unit="sample", file=sys.stdout):
        fr, y = ds[i]                                # fr: [T, p, H, W]
        flat = (np.asarray(fr).reshape(T, full_C) > 0).astype(np.uint8)   # binarise
        out[i] = flat[:, :n_channels]
        Y[i] = int(y)
    return torch.from_numpy(out), torch.from_numpy(Y)

def nmnist_cached(n_channels, T):
    cache = os.path.join(CFG["DATA_DIR"],
                         f"nmnist_{n_channels}_{T}_{CFG['NMNIST_TRAIN']}_{CFG['NMNIST_TEST']}.pt")
    if os.path.exists(cache):
        d = torch.load(cache)
        print(f"  [N-MNIST cache hit] train {tuple(d['Xtr'].shape)} test {tuple(d['Xte'].shape)}", flush=True)
        return d["Xtr"], d["Ytr"], d["Xte"], d["Yte"]
    import tonic
    sensor = tonic.datasets.NMNIST.sensor_size       # (34,34,2)
    print(f"  [N-MNIST cache miss] building {n_channels}ch x {T} frames -- downloads ~1GB first time", flush=True)
    t0 = time.time()
    tr = tonic.datasets.NMNIST(save_to=CFG["DATA_DIR"], train=True)
    te = tonic.datasets.NMNIST(save_to=CFG["DATA_DIR"], train=False)
    # Subsample: the full 60k set builds a 3.5 GB tensor that exhausts Colab RAM.
    # 10k/2k is ample -- operation counts are per-inference and unaffected by set size.
    Xtr, Ytr = _build_nmnist_frames(tr, T, n_channels, sensor,
                                    desc="N-MNIST train -> frames", limit=CFG["NMNIST_TRAIN"])
    Xte, Yte = _build_nmnist_frames(te, T, n_channels, sensor,
                                    desc="N-MNIST test  -> frames", limit=CFG["NMNIST_TEST"])
    torch.save({"Xtr": Xtr, "Ytr": Ytr, "Xte": Xte, "Yte": Yte}, cache)
    print(f"  [N-MNIST cached] in {time.time()-t0:.0f}s", flush=True)
    return Xtr, Ytr, Xte, Yte

# ---------------- loaders (mirror shd_loaders / hd_loaders) ----------------
NMNIST_CH = 2312          # full flattened N-MNIST channels (34*34*2)
NC = 10                   # MNIST / N-MNIST have 10 classes (audio digits had 20)

def _synth10(n_ch, spike_input, T=None, batch=None):
    """Synthetic fallback with the correct 10-class label range for vision.
    (The shared synth_loaders() hardcodes CFG['N_CLASSES']=20, which would make
    CrossEntropyLoss see labels a 10-output model cannot represent.)"""
    T = T or CFG["T"]; batch = batch or CFG["BATCH"]
    tr = SyntheticDS(CFG["SYN_TRAIN"], T, n_ch, NC, spike_input, seed=0)
    te = SyntheticDS(CFG["SYN_TEST"],  T, n_ch, NC, spike_input, seed=1)
    L = lambda d, s: DataLoader(d, batch_size=batch, shuffle=s, drop_last=False)
    return L(tr, True), L(te, False)

def mnist_loaders(T=None, batch=None):
    T = T or CFG["T"]; batch = batch or CFG["BATCH"]
    if CFG["USE_SYNTHETIC"]:
        return _synth10(784, spike_input=False, T=T, batch=batch)
    Xtr, Ytr, Xte, Yte = mnist_cached(T)
    L = lambda X, Y, s: DataLoader(MNISTMem(X, Y, T), batch_size=batch, shuffle=s)
    return L(Xtr, Ytr, True), L(Xte, Yte, False)

def nmnist_loaders(n_channels=NMNIST_CH, T=None, batch=None):
    T = T or CFG["T"]; batch = batch or CFG["BATCH"]
    if CFG["USE_SYNTHETIC"]:
        return _synth10(n_channels, spike_input=True, T=T, batch=batch)
    Xtr, Ytr, Xte, Yte = nmnist_cached(n_channels, T)
    L = lambda X, Y, s: DataLoader(SHDMem(X, Y), batch_size=batch, shuffle=s, num_workers=0)
    return L(Xtr, Ytr, True), L(Xte, Yte, False)

print("MNIST / N-MNIST loaders ready  |  mode:",
      "SYNTHETIC" if CFG["USE_SYNTHETIC"] else "REAL DATA")

### 7.2 — Run the four core experiments on the vision pair

The same four architectures as Sections 1–4, now on MNIST / N-MNIST. Note `n_out = 10` (ten digits, versus twenty for the bilingual audio digits). Results append to the same `RESULTS` list, tagged with a `V·` prefix so the vision runs are easy to separate from the audio ones in the final table.

- **V1 · MNIST → pure ANN** — the vision baseline (real-valued, all MACs).
- **V2 · N-MNIST → pure SNN** — genuinely event-native first layer.
- **V3 · N-MNIST → SNN-front → ANN-back** — the cheap hybrid (readout runs once).
- **V4 · MNIST → ANN-front → SNN-back** — the expensive hybrid; this is the "feed MNIST to an ANN, convert to spikes, then SNN" case, made concrete.

In [ ]:
if not CFG["RUN_VISION"]:
    print("RUN_VISION is False -> skipping (set it True in the config to run Section 7)")
else:
    NC = 10   # MNIST / N-MNIST have 10 classes
    
    # V1 · MNIST -> pure ANN  (real-valued input, no spiking neurons)
    mn_tr, mn_te = mnist_loaders()
    run("V1 MNIST -> ANN",
        lambda nu: ModelANN(784, CFG["HIDDEN"], NC, CFG["T"], neuron=nu, spike_input=False),
        mn_tr, mn_te, neurons=["-"])

In [ ]:
if not CFG["RUN_VISION"]:
    print("RUN_VISION is False -> skipping (set it True in the config to run Section 7)")
else:
    # V2 · N-MNIST -> pure SNN  (event-native input -> layer 1 is ACs)
    nm_tr, nm_te = nmnist_loaders(NMNIST_CH)
    run("V2 NMNIST -> SNN",
        lambda nu: ModelSNN(NMNIST_CH, CFG["HIDDEN"], NC, CFG["T"], neuron=nu, spike_input=True),
        nm_tr, nm_te)

In [ ]:
if not CFG["RUN_VISION"]:
    print("RUN_VISION is False -> skipping (set it True in the config to run Section 7)")
else:
    # V3 · N-MNIST -> SNN front -> ANN back  (cheap hybrid: readout runs ONCE)
    run("V3 NMNIST -> SNNfront+ANNback",
        lambda nu: ModelSNNfront(NMNIST_CH, CFG["HIDDEN"], NC, CFG["T"], neuron=nu, spike_input=True),
        nm_tr, nm_te)

In [ ]:
if not CFG["RUN_VISION"]:
    print("RUN_VISION is False -> skipping (set it True in the config to run Section 7)")
else:
    # V4 · MNIST -> ANN front -> SNN back  (expensive hybrid: dense front runs T times)
    #      This is exactly "feed MNIST to an ANN, convert to spikes, run SNN layers".
    run("V4 MNIST -> ANNfront(per-step)+SNN",
        lambda nu: ModelANNfront(784, CFG["HIDDEN"], NC, CFG["T"], neuron=nu, spike_input=False, collapse=False),
        mn_tr, mn_te)
    run("V4b MNIST -> ANNfront(collapsed)+SNN",
        lambda nu: ModelANNfront(784, CFG["HIDDEN"], NC, CFG["T"], neuron=nu, spike_input=False, collapse=True),
        mn_tr, mn_te)

### 7.2b — The stronger (recurrent) SNN on the vision pair

The same fairness argument applies here: the vision spiking models should be compared to the MNIST ANN at *comparable* accuracy, not at whatever accuracy they happen to land on. So we run the identical recurrent SNN on N-MNIST.

**But note the asymmetry that makes this interesting.** Recurrence buys memory across timesteps. On SHD that memory is genuinely useful, because speech unfolds in time. On N-MNIST the underlying digit is **static** — the time axis exists only because the camera saccades across a still image. So recurrence has less real temporal structure to exploit here, and we should expect it to help **less** than it did on audio. If that is what the numbers show, it is direct evidence for the project's deepest hypothesis: that spiking (and temporal machinery generally) pays off in proportion to how genuinely temporal the signal is.

In [ ]:
if not CFG["RUN_VISION"]:
    print("RUN_VISION is False -> skipping (set it True in the config to run Section 7)")
else:
    # Stronger recurrent SNN on N-MNIST (compare against V1 ANN and V2 plain SNN).
    run("V5 NMNIST -> RecurrentSNN+ANNback",
        lambda nu: ModelSNNRecurrent(NMNIST_CH, CFG["HIDDEN"], NC, CFG["T"],
                                     neuron=nu, spike_input=True),
        nm_tr, nm_te)

### 7.3 — Vision results, and the operation mix

In [ ]:
import pandas as pd
dfv = pd.DataFrame([r for r in RESULTS if r["model"].startswith(("V1","V2","V3","V4"))])
cols = ["model","neuron","accuracy_mean","accuracy_std","MACs","ACs","lif_steps",
        "N_syn","N_neur","syn_per_neur","E_opcount_fp32_nJ","E_memory_norm"]
cols = [c for c in cols if c in dfv.columns]
pd.set_option("display.width", 200, "display.max_columns", 40)
try:
    from IPython.display import display; display(dfv[cols])
except Exception:
    print(dfv[cols].to_string(index=False))

### 7.4 — Do the conclusions hold across both dataset pairs? (the hypothesis test)

*This is the point of the whole section. The project's hypotheses were formed on audio. Here we check each one against the vision pair, and read the contrast.*

In [ ]:
# Build a compact audio-vs-vision comparison keyed on the four analogous roles.
import pandas as pd
df_all = pd.DataFrame(RESULTS)

def pick(prefix):
    m = df_all[df_all["model"].str.startswith(prefix)]
    return m.iloc[0] if len(m) else None

roles = [
    ("Pure ANN (baseline)",        "S1", "V1"),
    ("Pure SNN (event-native)",    "S2", "V2"),
    ("SNN-front -> ANN-back",      "S3", "V3"),
    ("ANN-front -> SNN-back",      "S4a","V4 "),
]
print(f"{'role':30s} {'audio E_mem':>14s} {'vision E_mem':>14s}   {'audio acc':>9s} {'vision acc':>10s}")
print("-"*84)
rows = []
for name, sa, sv in roles:
    a, v = pick(sa), pick(sv)
    if a is None or v is None:
        print(f"{name:30s}  (missing a run: {sa}={a is not None}, {sv}={v is not None})")
        continue
    print(f"{name:30s} {a['E_memory_norm']:>14,.0f} {v['E_memory_norm']:>14,.0f}   "
          f"{a['accuracy_mean']:>9.3f} {v['accuracy_mean']:>10.3f}")
    rows.append((name, a, v))

# Which architecture is cheapest within each dataset? (the boundary-placement claim)
def cheapest(prefixes):
    subset = [pick(p) for p in prefixes]
    subset = [s for s in subset if s is not None]
    return min(subset, key=lambda r: r["E_memory_norm"])["model"] if subset else "n/a"
print()
print("Cheapest architecture (memory-inclusive):")
print("   audio :", cheapest(["S1","S2","S3","S4a"]))
print("   vision:", cheapest(["V1","V2","V3","V4 "]))

### 7.5 — The four hypotheses, judged against the data

> The version of this section I wrote before running deliberately did not pre-write its answers.
> Here they are. **Two predictions were wrong**, and one of the wrong ones turned out to be the
> most useful thing I learned.

#### H1 — *"Boundary placement is the lever."*  *my reading: supported — but not by the test I designed*

The comparison I intended — S3 vs S4a on audio, V3 vs V4 on vision — **cannot be used**, and I
should have caught this before running it. Every SNN-front model runs on the *spike* dataset and
every ANN-front model on the *real-valued* one, so that comparison is always cross-dataset **and**
cross-width — precisely what Section 5 forbids. The §7.4 cell's printed line
(*"cheapest: audio S4a, vision V3"*) is an **artifact of that confound, not a finding.** Read it as
a bug in my experimental design.

What *does* test H1 cleanly is the pair I built for another purpose: **S4a vs S4b** (audio,
**7.09×**) and **V4 vs V4b** (vision, **23.34×**). Same dataset, same width, accuracies
indistinguishable within one standard deviation. The only thing that changes is whether the dense
layer sits inside or outside the time loop — and layer-1 MACs differ by exactly `T = 25` in both
cases, as the structural self-test asserts.

**So on the two fully-controlled, iso-accuracy comparisons available to me, H1 holds on both
dataset pairs.** I think that makes it the strongest thing I have — but decide that for yourself
after looking at the ledger in §6.4, and note that "strongest available" is not the same as
"proven".

#### H2 — *"The event-native advantage is contingent on sparsity."*  *my reading: the principle survives, my prediction did not*

I predicted N-MNIST would be *denser* than SHD: *"event cameras are dense during a saccade, so its
N_spikes/syn may well sit above the break-even."*

Measured: **N-MNIST 1.06, SHD 5.46.** N-MNIST is **5.2× sparser**. I had the direction exactly
wrong.

The mechanism, once you look, is not subtle:

> **An event camera is a differentiator. A cochlea model is an integrator.** A pixel emits an event
> only when its brightness *changes*, so during a saccade only the moving *edges* of the digit
> fire and the rest of the frame stays silent. A cochlea channel fires whenever its frequency is
> *present*, so a sustained vowel keeps its channels firing frame after frame.

*"Event-based"* describes the **encoding**, not the **density** — and I should not have assumed I
could predict sparsity from the word "neuromorphic". The principle survives intact: sparsity is
what decides.

#### H3 — *"The leak's share tracks N_syn / N_neur."*  *my reading: the formula holds, the shorthand does not*

Cross-dataset, the ratio alone gives the wrong answer: V2 has a **3.5× higher** ratio than S2
(2154 vs 623) and yet a **higher** leak share (0.79% vs 0.53%).

The full §2.5 expression, which divides by `N_spikes/syn` as well, gets it right:

| | `lif_steps / ACs` | predicted | measured leak share |
|---|---|---|---|
| S2 (SHD) | 0.00735 | — | 0.53% |
| V2 (N-MNIST) | 0.01096 | **1.49× higher** | 0.79% = **1.49× higher** |

**Agreement to 0.2%.** The mechanism is confirmed; what fails is reading the leak off the *ratio
column alone*. §2.5's table should carry that correction, because the ratio is misleading whenever
spike rates differ between rows.

#### H4 — *"Spiking pays off in proportion to how temporal the signal is."*  *my reading: right answer, wrong reason → §7.6*

This one needs its own section, and it changes how the whole project should be read.

### 7.6 — The T-tax: why the vision headline is not what it looks like

#### The mistake was in my own baseline

`MNISTMem.__getitem__` returns `self.X[i].unsqueeze(0).expand(self.T, -1)` — **25 identical copies
of one image**. `ModelANN` then loops over all 25, computes the same logits every time, sums them,
and divides by 25. **V1 does 25× redundant work by construction.**

On HD that is entirely legitimate: those 25 frames are 25 *different* mel-spectrogram slices, and
an audio classifier genuinely has to look at all of them. **On MNIST it is pure waste** — an
artifact of my harness giving every model the same time axis, not a property of the task.

So the honest question is what happens when the MNIST ANN runs **once**, as any sane
implementation would:

| MNIST ANN baseline | Energy | Verdict vs the V2 SNN (5,181,165) |
|---|---|---|
| **V1 as I ran it** — 784 inputs, 25 identical passes | 57,422,080 | SNN **11.08× better** |
| Same network, run **once** | 2,296,883 | **SNN 2.26× WORSE** |
| Matched width (2,312 in), run once | 6,717,082 | SNN 1.30× better |
| Matched width, run once, **ideal reuse** | 3,285,672 | **SNN 1.74× WORSE** |

> **The 11× vision win is mostly my ANN baseline being made to do nothing twenty-four times.**
> Corrected, the vision SNN wins narrowly against a naive ANN and loses against a well-designed
> one. I would rather report this than let a number stand that I know is an artifact.

**An independent check that the correction is right.** The break-even for a *single-pass* ANN is
1.38 spikes/synapse. V2 measures **1.06**. The ratio is **1.31** — which is the 1.30× in the third
row, arrived at by completely separate arithmetic. Two routes, same answer.

#### What this does to H4 — it is right, for the opposite reason

H4 predicted that spiking pays off in proportion to how genuinely temporal the signal is. I
expected the mechanism to be that the SNN's temporal machinery has something to exploit.

**The prediction looks right. The mechanism looks like the reverse of what I assumed.**

> **Temporality does not help the spiking network. Temporality is what forces the *ANN* to make T
> passes — and the ANN's T-pass tax is the entire source of the SNN's headroom.**

Walk it through. On genuinely temporal data the ANN *must* process every frame, so it pays 25× and
the SNN's budget is `T × 1.38 = 34.5` spikes per synapse — generous. On static data the ANN should
process the input once, so the SNN's budget collapses back to `1.38` — ten times harder to clear,
purely because the competitor stopped being handicapped.

That one mechanism explains both dataset pairs at once, with no special pleading:

| | **HD / SHD** (audio) | **MNIST / N-MNIST** (vision) |
|---|---|---|
| Is the ANN's T-tax legitimate? | **Yes** — 25 distinct frames | **No** — one static image |
| Applicable break-even | **34.5** spikes/syn | **1.38** spikes/syn |
| Measured SNN sparsity | 5.46 (11.23 at 40 ch) | 1.06 |
| Fraction of budget used | 16% | 77% |
| Verdict, naive ANN, matched width | SNN **2.99× better** | SNN **1.30× better** |
| Verdict, ideal-reuse ANN, matched width | SNN **1.36× better** | SNN **1.74× WORSE** |

Notice what the last row does. **The two energy models flip the verdict on the vision pair exactly
as they flipped it on audio** — the project's headline finding, replicated on an independent
dataset pair that was chosen for a completely different reason.

#### Two caveats that soften the vision result further

**N-MNIST's sparsity is partly an illusion.** It has 2,312 input channels (34 × 34 × 2), but the
digit occupies a small part of that grid, so a large fraction of those channels **never fire at
all**. Much of the measured sparsity is *spatial dead channels*, not event-driven quiet — and
spatial zero-sparsity is exactly what the paper's activation-sparsity ANN model (Eq. 12, its
58%-zeros case, target 0.15) is built to exploit. A zero-skipping ANN would claw back much of it.

**The accuracy figure needs checking before I present it.** V2 reaches 0.9995 and V5 0.999, with
two seeds at exactly 1.000 — above typical published N-MNIST results, and above my own MNIST ANN,
even though N-MNIST is MNIST *plus* camera noise. Two candidate explanations, and I have not
established which applies:

- **Capacity.** V2 has 297,216 weights to V1's 101,632 — nearly 3× the parameters.
- **Subsampling.** `limit=` takes the **first** 10,000 train / 2,000 test samples rather than a
  random draw. If tonic's N-MNIST is stored in any non-random order, that changes the task.

Print the class histogram of the subsample before quoting 0.9995 to anyone.

---
## Comprehensive summary — what I think this whole project establishes

*This reads back the numbers my run produced. Each claim is tied to the mechanism that produces it, and flags the column it depends on — so please don't take a claim as true before you've looked at the run.*

### The through-line

Every result traces to one fact from §2.1: **on digital SRAM hardware, reaching memory costs ~5.4× a MAC, so ~99% of energy is memory traffic, not arithmetic.** The MAC→AC swap the field advertises is worth only 0.87 units; the SNN's real edge — one fewer memory fetch per synaptic op (§2.2) — is worth 5.4, about 6.2× more. Any story told only in MACs and ACs (Model a) is describing ~1% of the picture; Model b is the defensible one *for this class of hardware*.

### What the sections, together, show

1. **Boundary placement, not spiking per se, is the lever.** Sections 3 and 4 use the *same* ANN and SNN pieces. The only difference is which side of the accumulator the dense layer sits on. SNN-front → ANN-back (§3) pays its dense readout **once** and gives its output layer no membrane; ANN-front → SNN-back (§4a) pays its dense front end **`T` times** (my counting cell pins this ratio at exactly `T = 25`). That single structural choice is the difference between the cheap hybrid and the expensive one — the concrete content of Dampfhoffer's "SNN where sparse, ANN where dense" rule **[READ]**.

2. **The event-native advantage is real but narrow.** Only Section 2's first layer is genuinely spike-driven (ACs, not MACs), because only SHD arrives as spikes (§2.7). And it only pays off while the measured `N_spikes/syn` stays under the break-even thresholds of §2.3 — I check that column; the advantage is conditional, not automatic. On a good ANN accelerator the target is as low as 0.15–0.42 spikes/syn, which real SNNs frequently miss.

3. **The LIF leak is a structural tax whose size is set by the architecture.** Because I use LIF (the hardware-realistic choice, §2.4), every spiking neuron pays `11.8` per timestep. Its share of the budget tracks `N_syn/N_neur` exactly as §2.5 predicts: small at SHD's native ratio of 623 (§2), dominant at the matched 40-channel ratio of 52 (§5). This is also *why* the SNN-front hybrid (§3) is cheap: moving the output layer into ANN-land deletes its membrane entirely.

4. **The realistic prize is 1.2×–2.45×, not orders of magnitude.** That's the honest range from the estimated-energy hybrid literature (Louis et al. **[ABSTRACT]**; Dampfhoffer's worked hybrid gives 1.2× **[READ]**). The "100×–1000×" figures elsewhere compare a neuromorphic ASIC to a general-purpose GPU and mostly measure *that* gap, not spiking.

5. **On compute-in-memory, much of this inverts.** My whole "memory dominates, SNN skips a fetch" argument is a *von-Neumann-SRAM* argument (§SRAM-vs-CIM). On CIM the fetch is abolished for the ANN too, the bottleneck moves to the ADC, and the honest modern comparison becomes INT8-ANN-on-CIM vs SNN-on-CIM — a different, harder study I deliberately did not fake (§CIM-next-step). And the CIM actually shipping today is digital-SRAM CIM *for ANNs*, not analog CIM for SNNs.

### The accuracy caveat, stated plainly

The spiking models in Sections 2–5 land well below the pure ANN's accuracy. **That materially weakens any head-to-head energy claim involving them**, because energy at unequal accuracy is not a fair comparison. Two things follow, and both belong in any write-up:

- The **structural** findings (operation-count ratios, the once-vs-`T` effect, the two energy models disagreeing) do **not** depend on accuracy and stand regardless.
- The **cross-model energy comparisons** should be read off the accuracy-vs-energy frontier, and preferentially from the **recurrent SNN runs (Sections 5b / 7.2b)**, which were added specifically to bring spiking accuracy closer to the ANN's so the comparison is fairer. Report the accuracy next to every energy figure.

### Hard limits on what any of this proves

- **Energy is modelled, not measured.** Operation and spike counts are exact; joules are not. No hardware was involved. This notebook is one input to a decision, to be checked against real measurement — not a final authority.
- **Even the model is partial.** It excludes static/idle power (neuromorphic chips draw tens of mW even idle), inter-chip communication, and CIM entirely (§CIM-next-step — a different world where the ADC dominates).
- **Two front ends go uncounted, and both cut against the SNN.** The cochlea model behind SHD and the FFT behind HD both cost real energy that begins *before* the first synapse, where every published model starts.
- **The constants are ~2008-era 45 nm silicon.** Absolute joules on 2 nm-class silicon would differ substantially. I quote *ratios*, which are far more robust to a node change.
- **The fair comparator is a `ceil(log₂(T+1))`-bit ANN, not FP32** (§2.6). I show both energy models partly to keep that assumption honest.
- **Cross-dataset claims come only from Section 5.** Sections 1–4 have mismatched input widths (700 vs 40).

### If I were writing this up for a fair

I'd say, in order: (1) these are estimates, echoing the estimate-based nature of most of the field; (2) the finding is about *boundary placement*, replicating Dampfhoffer's rule on a new task, not a new architecture; (3) the numbers are ratios under ~45 nm INT8 assumptions; (4) the hybrid subfield is crowded — I'm replicating and clarifying, and "my searches didn't find an identical SHD sweep" is weaker evidence than "nobody did it"; (5) on the hardware inference is actually heading toward (CIM), the comparison partly inverts, and I've said so rather than hidden it. On anything a judge will lean on, verify against a primary source independently; treat this notebook as one carefully-audited input, not the last word.


---
## Appendix — how to read these results, and what not to claim

**Legitimate conclusions**
- Architecture rankings *within* one dataset (Sections 1–4 separately).
- IF vs LIF at fixed architecture — a clean single-variable comparison.
- Whether the two energy models agree; disagreement is a finding, not a nuisance.
- Cross-dataset comparison **only** from Section 5, where input width is matched.

**Not supported by this notebook**
- Absolute joules on modern hardware. Constants are ~45 nm silicon from ~2008; leading edge is now 2 nm-class.
- Anything about specific neuromorphic chips. Nothing here was measured on hardware — these are estimates, as are most published SNN numbers.
- Cross-dataset claims from Sections 1–4 (SHD has 700 inputs, HD has 40).
- Wall-clock timing as architecture evidence. Simulated SNNs are slow on GPUs — a property of the simulator.

**Uncounted costs, all cutting against the SNN**
- The cochlea model behind SHD and the FFT behind HD's spectrograms. Both are real energy; no model here counts either.
- Static/idle power. Neuromorphic chips draw non-trivial idle current.
- Inter-chip data movement, and the ADC costs that dominate compute-in-memory.

**Sanity checks before trusting a run**
1. Is the SHD input actually binary? (Setup cell 5 prints this in synthetic mode; the structural test cell asserts it.) If not, layer 1 was counted as MACs and every SHD number is wrong.
2. Does the pure ANN report exactly zero ACs? It must.
3. Is Section 3's readout MAC count a one-off (`B×syn2`), well below a per-timestep tax?
4. Is the between-seed spread smaller than the between-model spread? If not, you have noise, not a result.

*Built July 2026. Energy figures are estimates under ~45 nm assumptions; treat every **[UNREAD]** figure as requiring independent confirmation, and treat this notebook as one audited input to a decision, not the last word.*

---
# Appendix A · Compute-in-memory, and why it is out of scope

> Moved here from the main line. CIM is the **one** hardware change that *inverts* my argument
> rather than scaling it (§2.0), so it belongs in the record — but it interrupted the results
> narrative where it used to sit, and it was being explained three separate times. The three
> passages below are the original text, consolidated.

---
## Interlude — SRAM vs Compute-in-Memory, for ANNs and SNNs  *(point 9, because I kept confusing these myself)*

Everything in my energy model (§2) assumes a **von Neumann digital accelerator**: compute and memory are separate boxes, and you shuttle data between them. But two other hardware ideas keep coming up, and they matter for how honest my comparison is, so here's the plain-language version.

**SRAM (what my model assumes).** Fast on-chip memory that you still *fetch from*. Compute and memory are separate; every operation moves data back and forth. This is where the "one SRAM access = 5.4 MACs" cost (§2.1) comes from. The whole SNN advantage in my model is "a spike skips one of those fetches."

**CIM / Compute-in-Memory.** Do the multiply *inside* the memory array — the data never travels to a separate compute unit. Two flavours:
- **Digital-SRAM CIM** (e.g. d-Matrix Corsair, shipping 2026): the in-memory-compute array is built from ordinary SRAM cells. Stable, digital, no analog noise. Currently used for **transformer/LLM inference** — an ANN technology.
- **Analog-crossbar CIM** (RRAM / PCM / memristor): weights are stored as physical conductances; Ohm's and Kirchhoff's laws do the multiply-accumulate; an **ADC** reads the column current. Denser and potentially lower-energy, but noisy, and still mostly pre-commercial.

Here's the table I wish I'd had at the start:

| | Von Neumann + SRAM (my model, §2) | Compute-in-Memory (CIM) |
|---|---|---|
| Where the multiply happens | Separate compute unit; weights fetched | *Inside* the memory array; weights never move |
| Dominant energy cost | **Memory fetch** (the 5.4×) | **ADC conversion** + communication |
| Why an SNN helps | Skips one fetch per synapse (§2.2) | Binary activations → very low ADC precision |
| Data reuse (ANN's big win, §2.3) | Decisive | Mostly irrelevant — weights are already stationary |
| Who it's mainly built for today | Both | **ANNs** (DNN/transformer inference) |
| SNN's worst liability | Membrane *traffic* (the leak, §2.4) | Membrane *area* (LIF cache >1000× a ReLU) |

**The key realisation for my project:** my whole energy argument — "memory dominates, and the SNN skips a fetch" — is a statement about **von Neumann + SRAM hardware**. On CIM the fetch is abolished *for everyone*, so that specific SNN advantage largely disappears and the comparison changes character entirely. That's the setup for the next section.


### Compute-in-memory (CIM / IMC) — and the surprising fact that it's mostly an *ANN* story

CIM does the matrix multiply *inside* the memory array — inputs become voltages, weights are stored conductances, physics does the multiply-accumulate, and an **ADC** reads the result. This abolishes weight movement, the dominant cost in §2's digital story.

Here's the fact that reshaped how I think about this project: **CIM is overwhelmingly aimed at ANNs, not SNNs.** Vector-matrix multiply is "the compute kernel of deep learning," and the flagship parts are ANN inference accelerators:

- **IBM's 64-core phase-change-memory chip** (Nature Electronics, 2023) — a mixed-signal *ANN* inference chip ([survey context](https://arxiv.org/abs/2401.14428) **[READ]**).
- **d-Matrix Corsair** — [full production June 2026](https://www.d-matrix.ai/announcements/d-matrix-corsair-ai-inference-platform-enters-full-production-to-meet-customer-demand/) **[READ]**, on TSMC 6 nm. Crucially it's **digital in-memory compute built on SRAM**, not analog crossbars, and it targets **LLM/transformer inference**. Its whole pitch is dodging the DRAM bottleneck by keeping weights in on-chip SRAM ([CNBC](https://www.cnbc.com/2026/06/09/nvidia-d-matrix-chip-production-microsoft.html) **[READ]**).
- Analog RRAM/PCM crossbar CIM (the kind that makes *SNNs* look 26× better in simulation) is, per the [2026 patent landscape](https://www.patsnap.com/resources/blog/articles/in-memory-analog-computing-landscape-2026/) **[READ]**, still pre-commercial and moving toward "analog-compute / digital-readout hybrids" to manage noise — and its dominant application is *DNN inference*, i.e. ANNs.

So the commercial CIM that's actually shipping is **digital-SRAM CIM for transformers (an ANN technology)**, while **analog-crossbar CIM for SNNs is a research niche**. That gap between the simulated-analog-SNN story and the shipping-digital-ANN reality is exactly why I trust *ratios and mechanisms* over any single headline multiplier — and it's central to my "what's a fair next step" discussion (§6-CIM).

> **Do not compare** the simulated CIM "26×" figures to my "1.2–2.45×" hybrid figures. Different baselines, platforms, and metrics. They describe different worlds; Dampfhoffer explicitly place CIM outside their scope, and so does my energy model.

---
## What a fair "next-step" comparison would look like on modern hardware  *(point 8)*

I want this project to be as close to the real world as an honest student replication can be, so I have to say clearly where my own comparison stops being the right one.

**My comparison (this whole notebook) is a digital-von-Neumann comparison.** Its central claim — "memory dominates, and the SNN saves one fetch per synapse because a spike is its own address" (§2.2) — is *true on the hardware it assumes*, and that hardware (SRAM + separate compute) is real and widely deployed. But it is **not** the only hardware, and it may not be the future one.

**Here's the honest tension, and it's important enough that I don't want to bury it.** From §4, the compute-in-memory hardware actually shipping in 2026 (d-Matrix Corsair) is **digital-SRAM CIM built for ANNs/transformers**, and analog-crossbar CIM — the kind that makes *SNNs* look dramatically better in simulation — is still a research niche. So the most likely near-future "apples-to-apples" is not "SNN vs ANN on a GPU" but:

> **INT8 ANN-on-CIM  vs  SNN-on-CIM, on the same crossbar, counting ADC + area + timestep-noise.**

And on *that* comparison, my project's core advantage weakens, for reasons I can state but not measure here:

- **The fetch I credited the SNN for skipping (§2.2) is abolished for the ANN too** on CIM, because the ANN's weights also stop moving. The SNN's structural edge in my model is a von-Neumann artefact.
- **On CIM the SNN wins for a *different* reason** — binary activations allow very low ADC precision, and ADC cost dominates. That's a real advantage, but it's not the one my notebook measures.
- **The SNN also picks up a new liability on CIM**: the LIF membrane needs a physical cache that can cost >1000× the area of an ANN's ReLU comparator, and analog non-idealities accumulate over timesteps, capping useful `T` at around 4 (SpikeSim / Moitra et al. **[READ]** in the primer).

**Why I did NOT try to compute the CIM comparison.** It needs specialised simulators (SpikeSim, NeuroSim / CrossSim), models device physics I can't validate, and would take its own multi-week project — and any number I produced would be a simulator estimate stacked on assumptions I can't check. Manufacturing a CIM number just to have one would violate the honesty rule this whole project runs on. So I state the mechanism and the direction, and stop.

**What I'd tell someone extending this project:** the truly "modern, practical" comparison of hybrids is on CIM, and it is a *different and harder* study than mine. My notebook is the correct comparison for **digital SRAM accelerators**, which are real and shipping; the CIM comparison is the correct one for where inference hardware is heading, and it partly *inverts* my conclusions. Both are true; they describe different worlds. Knowing which world a claim belongs to is the whole game.


---
# Appendix B · PyTorch & snnTorch primer, and the LIF playground

> Moved here from the main line. This is reference material rather than argument — read it once
> if you need it, then go back to the models in Setup cell 6.

---
## A PyTorch & snnTorch mini-tutorial — read this once, refer back anytime

*If you already know PyTorch, skip this. It's written for someone fluent in Python and NumPy, who understands SNNs / LIF / surrogate gradients in theory, but hasn't used PyTorch or snnTorch. Every call used in this notebook is explained here, with NumPy analogies. It's meant to be reusable — keep it.*

### 1 · A tensor is a NumPy array that can differentiate itself

A `torch.Tensor` is almost exactly a NumPy `ndarray`: same indexing, broadcasting, `.shape`, `.mean()`, `.reshape()`. Two differences matter:

- **It can live on a GPU.** `x.to(DEV)` moves a tensor to the device (`"cuda"` or `"cpu"`). NumPy is CPU-only; this is why we write `.to(DEV)` everywhere.
- **It can track gradients.** If a tensor is part of a model's computation, PyTorch silently records every operation applied to it, building a graph so it can later compute derivatives automatically (this is *autograd*, section 4). A plain NumPy array can't do this.

Tensor calls you'll see, and their NumPy equivalents:

| PyTorch | NumPy equivalent | Note |
|---|---|---|
| `torch.zeros(a, b)` | `np.zeros((a,b))` | `dtype=torch.float32` sets the type |
| `torch.randn(...)` / `torch.rand(...)` | `np.random.randn` / `rand` | normal / uniform random |
| `torch.randint(0, n, (k,))` | `np.random.randint` | random integers |
| `x.mean(1)` | `x.mean(axis=1)` | PyTorch calls the axis `dim` |
| `x.reshape(a, -1)` | same | `-1` = "infer this dimension" |
| `x.squeeze(0)` / `x.unsqueeze(0)` | `np.squeeze` / `np.expand_dims` | remove / add a size-1 axis |
| `x.argmax(1)` | `x.argmax(axis=1)` | index of the max along a dim |
| `torch.unique(t)` | `np.unique` | used to check a tensor is 0/1 |
| `x.numpy()` | — | convert a (CPU) tensor back to a NumPy array |
| `torch.as_tensor(np_array)` | — | wrap a NumPy array as a tensor |
| `x.detach()` | — | make a copy that is **not** tracked by autograd (section 4) |

`torch.manual_seed(s)` and `torch.Generator().manual_seed(s)` are the torch versions of `np.random.seed` — they make random tensors reproducible.

### 2 · `nn.Linear` is a learnable `W @ x + b`

```python
fc = nn.Linear(n_in, n_out)   # creates a weight matrix W (n_out x n_in) and bias b, both learnable
y  = fc(x)                    # computes  y = x @ W.T + b
```

That's the whole thing. `nn.Linear(40, 128)` is a fully-connected layer from 40 inputs to 128 outputs — the `W` and `b` inside it are exactly the parameters gradient descent will train. In this notebook, `self.fc1` and `self.fc2` are the two weight layers of every model; **their `N_syn` = `n_in × n_out` is the synapse count our energy model charges for.**

`torch.relu(x)` is the ReLU nonlinearity, `max(0, x)`, applied elementwise — identical to `np.maximum(0, x)`.

`nn.functional` (imported as part of torch) holds stateless helper functions; we use `torch.nn.functional.interpolate` once, only to resample a spectrogram to `T` frames — it's not part of the neural network itself.

### 3 · `nn.Module` is the base class for "a thing with learnable parameters"

Every model in this notebook subclasses `nn.Module`. The pattern is always the same and worth memorising:

```python
class MyModel(nn.Module):
    def __init__(self, ...):
        super().__init__()               # REQUIRED first line
        self.fc1 = nn.Linear(a, b)        # define the layers as attributes
    def forward(self, x):                 # define the computation
        return self.fc1(x)
```

- Anything you assign to `self.` that is an `nn.Linear` (or an `snn.Leaky`) is automatically registered, so PyTorch can find all its weights later via `model.parameters()`.
- **You never call `forward` directly.** You call the model like a function — `model(x)` — and PyTorch runs `forward` for you (plus some bookkeeping). In our code, `train_eval` calls `model(xb)`; that triggers `forward`.
- `model.to(DEV)` moves all the model's weights to the GPU/CPU in one shot.
- `model.train()` and `model.eval()` flip the model between training and evaluation modes. (They matter for layers like dropout/batch-norm; our models don't use those, but it's correct practice — and we count operations only in `eval` mode.)

### 4 · Autograd and the training loop — the part with no NumPy analogy

This is the piece that's genuinely new if you've only done NumPy. In NumPy you'd derive gradients by hand. PyTorch does it automatically: as the forward pass runs, it records the graph of operations; then one call walks that graph backward applying the chain rule.

The five-line ritual — you'll see exactly this inside `train_eval`:

```python
opt = torch.optim.Adam(model.parameters(), lr=...)   # the optimiser: knows which tensors to update
lossf = nn.CrossEntropyLoss()                         # loss for classification (softmax + neg-log-likelihood)

opt.zero_grad()          # 1. clear old gradients (they accumulate by default, so you must reset)
loss = lossf(model(xb), yb)   # 2. forward pass -> prediction -> scalar loss
loss.backward()          # 3. autograd fills in every parameter's .grad via the chain rule
opt.step()               # 4. optimiser nudges each weight down its gradient
```

- `model.parameters()` hands the optimiser every learnable tensor (all the `fc1`/`fc2` weights and biases).
- `Adam` is a standard optimiser — think of it as gradient descent with per-parameter adaptive step sizes. `lr` is the learning rate.
- `CrossEntropyLoss` takes raw output scores (logits) of shape `[batch, n_classes]` and the integer labels, and returns one scalar. Minimising it = classifying better.
- `loss.backward()` is where autograd earns its keep: it computes `d loss / d weight` for **every** weight, automatically. This is the thing you'd otherwise do by hand.
- **`with torch.no_grad():`** turns off graph-tracking for a block. We wrap evaluation in it because we're only measuring, not training — it's faster and uses less memory.
- **`x.detach()`** does the same locally: it gives you a value with the gradient-tracking stripped off. We use it in the counting hooks (`float(t.detach().sum())`) so that reading a spike total for bookkeeping never accidentally entangles with the gradient graph.

### 5 · Feeding data: `Dataset` and `DataLoader`

- A `Dataset` is any object with `__len__` (how many samples) and `__getitem__(i)` (return sample `i` as `(x, y)`). Our `SHDWrap`, `TensorDS`, and `SyntheticDS` are all just this — thin wrappers that hand back one `(input, label)` pair at a time.
- A `DataLoader` wraps a `Dataset` and yields it in **batches**: `for xb, yb in loader:` gives you `xb` of shape `[batch, ...]`. It handles shuffling (`shuffle=True`) and batching for you. Think of it as a smart generator over your data.

### 6 · snnTorch: the spiking part

snnTorch adds spiking neurons that plug into ordinary PyTorch. We use exactly one neuron type and one surrogate gradient.

**`snn.Leaky(beta=..., spike_grad=...)`** is a Leaky-Integrate-and-Fire (LIF) layer.

- `beta` is the membrane decay per timestep (our `0.9`): each step the membrane potential is multiplied by `beta` before new input is added. `beta = 1.0` would mean *no* leak (that's the IF neuron; we don't use it — see section 2.4 of the intro).
- `spike_grad` is the **surrogate gradient function** used on the backward pass. The forward pass fires a hard 0/1 spike (a step function, whose true derivative is useless for training); on the backward pass snnTorch substitutes a smooth approximation so gradients can flow. We use `surrogate.fast_sigmoid()`, a standard choice — the derivative of a fast sigmoid, centred at the firing threshold.

**How you *run* a LIF layer — the two-line pattern:**

```python
mem = lif.init_leaky()          # create the initial membrane potential (a zero tensor of the right shape)
for t in range(T):              # step through time yourself
    spk, mem = lif(current, mem)   # feed input current + previous membrane -> (output spike, new membrane)
```

This is the crux, and it's why every model has a `for t in range(T)` loop. Unlike an ordinary layer that runs once, a spiking layer is **stateful over time**: you must (1) initialise its membrane with `init_leaky()`, then (2) call it once per timestep, passing the membrane back in each time. `spk` is the binary spike output at that step; `mem` is the updated membrane potential you feed into the next step.

- `lif(self.fc1(xt), mem)` means: take timestep `t`'s input `xt`, pass it through the weight layer `fc1` to get the input current, feed that plus the previous membrane into the LIF neuron, get back this step's spikes and the new membrane.
- The membrane potential is the "integrate" in integrate-and-fire; `init_leaky()` just sets it to zeros to start.

**That's the entire snnTorch surface used here:** `snn.Leaky`, `.init_leaky()`, `surrogate.fast_sigmoid()`, and calling the layer as `spk, mem = lif(x, mem)` inside a time loop. Everything else is ordinary PyTorch.

### 7 · Putting it together — how to read any model's `forward` below

Every model does the same three things; once you see the pattern, all five read quickly:

1. **Initialise membranes** for any spiking layers (`lif.init_leaky()`).
2. **Loop over `T` timesteps**, and at each step push the data through the layers — dense layers (`fc`, `relu`) run as normal, spiking layers use the `spk, mem = lif(...)` pattern.
3. **Accumulate an output** over time (sum or mean of the spikes/logits), and **tally operation counts** into `stats` for the energy model.

The only thing that differs between models is *which layers are inside the time loop and which are outside it* — and that, as the intro says, is the entire experiment.


---
## Optional — play with one LIF neuron before reading the models

*This cell is not part of the experiment — it's the standard first exercise from the snnTorch tutorials, and it's the fastest way to make the `spk, mem = lif(x, mem)` loop click. Run it, then read the models; skip it if the primer already made sense.*

I drive a **single** LIF neuron with a constant input current for a while, switch the current off, and record two things at every timestep: its **membrane potential** (the "integrate") and its **output spikes** (the "fire"). You should see the membrane ramp up, fire a spike and reset each time it crosses the threshold, then decay away once the input stops — the leak. Try changing `beta` (the decay) and `current` and re-running to build intuition.

In [ ]:
# Self-contained: only needs torch, snntorch, matplotlib (no dataset, no other cells).
import torch as _t
import snntorch as _snn
from snntorch import surrogate as _surrogate
import matplotlib.pyplot as _plt

# --- knobs to experiment with ---
_beta    = 0.9      # membrane decay per step: closer to 1 = leaks slower; 1.0 = no leak (IF)
_current = 0.25     # constant input current while "on"
_T       = 60       # total timesteps
_on_until = 40      # input current is on for the first 40 steps, then off

# one LIF neuron (snn.Leaky). spike_grad only matters for training; harmless here.
_neuron = _snn.Leaky(beta=_beta, spike_grad=_surrogate.fast_sigmoid())

_mem = _neuron.init_leaky()          # starting membrane potential (a zero tensor) -- primer 6
_mem_trace, _spk_trace = [], []
for _step in range(_T):
    _inp = _t.tensor(_current) if _step < _on_until else _t.tensor(0.0)  # input current this step
    _spk, _mem = _neuron(_inp, _mem)  # THE pattern: (current, prev membrane) -> (spike, new membrane)
    _mem_trace.append(float(_mem))
    _spk_trace.append(float(_spk))

# --- plot membrane potential and spikes ---
_fig, (_a1, _a2) = _plt.subplots(2, 1, figsize=(9, 4), sharex=True,
                                 gridspec_kw={"height_ratios": [3, 1]})
_a1.plot(_mem_trace, color="#3D6489"); _a1.axhline(1.0, ls="--", color="#E24B4A", lw=1)
_a1.set_ylabel("membrane\npotential"); _a1.legend(["membrane", "threshold = 1.0"], fontsize=8, loc="upper right")
_a1.axvspan(0, _on_until, color="#E8A33D", alpha=0.12)
_a2.eventplot([i for i, s in enumerate(_spk_trace) if s > 0], color="#1D9E75", lineoffsets=0.5)
_a2.set_yticks([]); _a2.set_ylabel("spikes"); _a2.set_xlabel("timestep")
_a2.set_xlim(0, _T)
_fig.suptitle(f"One LIF neuron  (beta={_beta}, input={_current} for first {_on_until} steps)", fontsize=10)
_plt.tight_layout(); _plt.show()

print(f"total spikes fired: {int(sum(_spk_trace))}")
print("Notice: membrane RAMPS while current is on (shaded), FIRES + resets at the threshold,")
print("then DECAYS away once the input stops -- that decay is the 'leak' in Leaky-I-F.")
print("Try beta=1.0 (no leak) or a bigger current, and re-run to see the behaviour change.")